# 25 · Autoencodeurs et VAE — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch25_autoencodeurs/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch25_autoencodeurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py 25`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 25.Q1 à 25.Q12, 25.R1 à 25.R3, 25.1 à 25.4, 25.6, 25.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 25.14 | Prédire : un autoencodeur entraîné sur une seule image | 🔮 | ★ | 15 |
|  | 25.15 | Premier autoencodeur dense 784-20-784 en PyTorch | 📦 | ★★ | 25 |
|  | 25.16 | Autoencodeur profond 512-256-128-20 | 📦 | ★★ | 25 |
|  | 25.17 | Prédire : ce qu'un autoencodeur de chiffres fait d'un vêtement | 🔮 | ★★ | 15 |
|  | 25.18 | Reproduire la carte latente 2D et sa grille décodée (fig. 25.29-25.30) | 🎨 | ★★ | 30 |
|  | 25.19 | Secouer les variables latentes : bruit global, une seule composante, latents aléatoires | 🔬 | ★★ | 25 |
|  | 25.20 | Interpoler entre deux codes : linéaire et sphérique | 🔨 | ★★ | 20 |
|  | 25.21 | Mélanger dans l'espace des pixels ou dans l'espace latent | 🔬 | ★★ | 25 |
|  | 25.22 | Taille du goulot et profondeur : la courbe d'erreur de reconstruction | 🔬 | ★★★ | 40 |
|  | 25.23 | L'autoencodeur linéaire face à l'ACP de scikit-learn | 🔬 | ★★★ | 40 |
| B | 25.24 | Autoencodeur convolutif à 147 latents | 📦 | ★★ | 30 |
|  | 25.25 | Le débruiteur qui apprenait le bruit | 🐛 | ★★ | 25 |
|  | 25.26 | Autoencodeur débruiteur (bruit gaussien, écrêtage) | 📦 | ★★ | 30 |
|  | 25.27 | Sur-échantillonnage explicite ou convolutions à pas : temps et qualité | 🔬 | ★★★ | 35 |
| C | 25.28 | La reparamétrisation en NumPy : passe avant et passe arrière | 🔨 | ★★ | 25 |
|  | 25.29 | Divergence KL gaussienne et son gradient | 🔨 | ★★ | 20 |
|  | 25.30 | Un VAE en PyTorch : reconstruction + KL | 📦 | ★★ | 30 |
|  | 25.31 | Prédire : décoder des tirages N(0, I) avec un AE puis avec un VAE | 🔮 | ★★ | 15 |
|  | 25.32 | Un VAE est stochastique : sorties répétées, bruit relatif, interpolations | 🔬 | ★★ | 30 |
|  | 25.33 | Reproduire le nuage latent d'un VAE 2D et la grille −3…3 (fig. 25.67-25.68) | 🎨 | ★★ | 30 |
|  | 25.34 | Diagnostiquer des courbes reconstruction/KL (effondrement du posterior) | 📈 | ★★ | 15 |
|  | 25.35 | Exporter le décodeur comme générateur autonome et le tester | 🛠️ | ★★ | 20 |
|  | 25.36 | β-VAE : échanger la netteté contre la régularité | 🔬 | ★★★ | 40 |
|  | 25.37 | Détecter des anomalies par l'erreur de reconstruction | 🔬 | ★★★ | 45 |
|  | 25.38 | Un VAE entièrement en NumPy avec mylearn | 🔨 | ★★★★ | 150 |
|  | 25.39 | Défi débruitage Fashion-MNIST : le meilleur PSNR sous contrainte | 🏆 | ★★★★ | 120 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="25")  # your own library

## Objectifs et rappel express

- Construire et entraîner en PyTorch des autoencodeurs denses, convolutifs et débruiteurs, et explorer leur espace latent (carte 2D, bruit, interpolation).
- Relier l'autoencodeur linéaire à l'ACP ; mesurer un débruitage (PSNR).
- Écrire en NumPy le cœur du VAE (`mylearn.vae`), entraîner un VAE, régler β, générer, détecter des anomalies.

**Rappel express.** $\mathbf{z} = f_\theta(\mathbf{x})$, $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$, la cible est l'entrée. Convolution transposée : $H_{out} = (H_{in} - 1)s - 2p + k + p_{out}$. PSNR $= 10 \log_{10}(1/\text{MSE})$ pour des pixels dans $[0, 1]$. VAE : $\mathbf{z} = \boldsymbol{\mu} + e^{\frac{1}{2}\log\boldsymbol{\sigma}^2} \odot \boldsymbol{\varepsilon}$, loss = reconstruction + $\beta$ KL, $\mathrm{KL} = \tfrac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 25.1 à 25.4 et 25.6, ∂ 25.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`) ; une forme ou un vecteur : entre guillemets (`"(3, 5, 1)"`, `"(1, -2)"`) ; un mot : entre guillemets (`"ABCD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 25.5 et 25.8 et la démarche de ∂ 25.7, les estimations 🧮 25.9 et 25.10, la réflexion (🗣️ 25.11, ⚖️ 25.12, 📄 25.13) et l'entretien se corrigent avec `05_solutions.md`.

**25.Q1 — Avec ou sans perte : le test de l'inversibilité**

In [ ]:
# 25.Q1: the values you found on paper
answer_25_Q1a = ...  # a) the letters, e.g. "AF"
answer_25_Q1b = ...  # b) True or False
answer_25_Q1c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q1d = ...  # d) a number
answer_25_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q1a, answer_25_Q1b, answer_25_Q1c, answer_25_Q1d, answer_25_Q1e]):
    wb.check(f"25.Q1{letter}", answer)

**25.Q2 — Pourquoi JPEG et MP3 battent un autoencodeur généraliste**

In [ ]:
# 25.Q2: the values you found on paper
answer_25_Q2a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q2b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q2c = ...  # c) True or False
answer_25_Q2d = ...  # d) the letters, e.g. "AF"
answer_25_Q2e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_25_Q2a, answer_25_Q2b, answer_25_Q2c, answer_25_Q2d, answer_25_Q2e]):
    wb.check(f"25.Q2{letter}", answer)

**25.Q3 — Mélange de contenu ou mélange de paramètres ?**

In [ ]:
# 25.Q3: the values you found on paper
answer_25_Q3a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q3b = ...  # b) a word as a string, e.g. "ABCD"
answer_25_Q3c = ...  # c) True or False
answer_25_Q3d = ...  # d) the letter of your choice, e.g. "F"

for letter, answer in zip("abcd", [answer_25_Q3a, answer_25_Q3b, answer_25_Q3c, answer_25_Q3d]):
    wb.check(f"25.Q3{letter}", answer)

**25.Q4 — Goulot, code, variables latentes : vrai ou faux**

In [ ]:
# 25.Q4: the values you found on paper
answer_25_Q4a = ...  # a) True or False
answer_25_Q4b = ...  # b) True or False
answer_25_Q4c = ...  # c) True or False
answer_25_Q4d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q4a, answer_25_Q4b, answer_25_Q4c, answer_25_Q4d, answer_25_Q4e]):
    wb.check(f"25.Q4{letter}", answer)

**25.Q5 — Le réseau qui ne savait redessiner qu'une image**

In [ ]:
# 25.Q5: the values you found on paper
answer_25_Q5a = ...  # a) a whole number
answer_25_Q5b = ...  # b) a whole number
answer_25_Q5c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q5d = ...  # d) True or False
answer_25_Q5e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q5a, answer_25_Q5b, answer_25_Q5c, answer_25_Q5d, answer_25_Q5e]):
    wb.check(f"25.Q5{letter}", answer)

**25.Q6 — Profondeur ou nombre de latents : qui améliore quoi ?**

In [ ]:
# 25.Q6: the values you found on paper
answer_25_Q6a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q6b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q6c = ...  # c) the letters, e.g. "AF"
answer_25_Q6d = ...  # d) True or False
answer_25_Q6e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q6a, answer_25_Q6b, answer_25_Q6c, answer_25_Q6d, answer_25_Q6e]):
    wb.check(f"25.Q6{letter}", answer)

**25.Q7 — Du bruit dans l'espace latent d'un autoencodeur classique**

In [ ]:
# 25.Q7: the values you found on paper
answer_25_Q7a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q7b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q7c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q7d = ...  # d) True or False
answer_25_Q7e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q7a, answer_25_Q7b, answer_25_Q7c, answer_25_Q7d, answer_25_Q7e]):
    wb.check(f"25.Q7{letter}", answer)

**25.Q8 — Lire une carte latente à deux dimensions**

In [ ]:
# 25.Q8: the values you found on paper
answer_25_Q8a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q8b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q8c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q8d = ...  # d) True or False
answer_25_Q8e = ...  # e) a vector as a string, e.g. "(1, -2)"

for letter, answer in zip("abcde", [answer_25_Q8a, answer_25_Q8b, answer_25_Q8c, answer_25_Q8d, answer_25_Q8e]):
    wb.check(f"25.Q8{letter}", answer)

**25.Q9 — Autoencodeur linéaire et ACP**

In [ ]:
# 25.Q9: the values you found on paper
answer_25_Q9a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q9b = ...  # b) True or False
answer_25_Q9c = ...  # c) True or False
answer_25_Q9d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q9e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_25_Q9a, answer_25_Q9b, answer_25_Q9c, answer_25_Q9d, answer_25_Q9e]):
    wb.check(f"25.Q9{letter}", answer)

**25.Q10 — Autoencodeurs convolutifs et débruiteurs**

In [ ]:
# 25.Q10: the values you found on paper
answer_25_Q10a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q10b = ...  # b) True or False
answer_25_Q10c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q10d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q10a, answer_25_Q10b, answer_25_Q10c, answer_25_Q10d, answer_25_Q10e]):
    wb.check(f"25.Q10{letter}", answer)

**25.Q11 — Une entrée jamais vue : un tigre chez les chiffres**

In [ ]:
# 25.Q11: the values you found on paper
answer_25_Q11a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q11b = ...  # b) True or False
answer_25_Q11c = ...  # c) True or False
answer_25_Q11d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q11e = ...  # e) the letters, e.g. "AF"

for letter, answer in zip("abcde", [answer_25_Q11a, answer_25_Q11b, answer_25_Q11c, answer_25_Q11d, answer_25_Q11e]):
    wb.check(f"25.Q11{letter}", answer)

**25.Q12 — Les trois exigences d'un VAE et l'astuce de reparamétrisation**

In [ ]:
# 25.Q12: the values you found on paper
answer_25_Q12a = ...  # a) the letters, e.g. "AF"
answer_25_Q12b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q12c = ...  # c) True or False
answer_25_Q12d = ...  # d) a number, 2 decimals
answer_25_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q12a, answer_25_Q12b, answer_25_Q12c, answer_25_Q12d, answer_25_Q12e]):
    wb.check(f"25.Q12{letter}", answer)

**25.R1 — Quelles augmentations garder pour des chiffres manuscrits ?**

In [ ]:
# 25.R1: the values you found on paper
answer_25_R1a = ...  # a) the letters, e.g. "AF"
answer_25_R1b = ...  # b) True or False
answer_25_R1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_25_R1a, answer_25_R1b, answer_25_R1c]):
    wb.check(f"25.R1{letter}", answer)

**25.R2 — Résumer une séquence entière dans l'état final d'un LSTM**

In [ ]:
# 25.R2: the values you found on paper
answer_25_R2a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_25_R2b = ...  # b) a number, 1 decimal
answer_25_R2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_25_R2a, answer_25_R2b, answer_25_R2c]):
    wb.check(f"25.R2{letter}", answer)

**25.R3 — La divergence KL, ce que mesure le terme de régularisation du VAE**

In [ ]:
# 25.R3: the values you found on paper
answer_25_R3a = ...  # a) a number, 2 decimals
answer_25_R3b = ...  # b) True or False
answer_25_R3c = ...  # c) True or False
answer_25_R3d = ...  # d) True or False
answer_25_R3e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_25_R3a, answer_25_R3b, answer_25_R3c, answer_25_R3d, answer_25_R3e]):
    wb.check(f"25.R3{letter}", answer)

**Ex 25.1 — Compter les paramètres d'un autoencodeur à goulot**

In [ ]:
# Ex 25.1: the values you found on paper
answer_25_1a = ...  # a) a whole number
answer_25_1b = ...  # b) a whole number
answer_25_1c = ...  # c) a whole number
answer_25_1d = ...  # d) a whole number
answer_25_1e = ...  # e) a whole number
answer_25_1f = ...  # f) a number, 1 decimal
answer_25_1g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_25_1a, answer_25_1b, answer_25_1c, answer_25_1d, answer_25_1e, answer_25_1f, answer_25_1g]):
    wb.check(f"25.1{letter}", answer)

**Ex 25.2 — Mélanger deux cercles : pixels contre paramètres**

In [ ]:
# Ex 25.2: the values you found on paper
answer_25_2a = ...  # a) a vector as a string, e.g. "(1, -2)"
answer_25_2b = ...  # b) a number, 2 decimals
answer_25_2c = ...  # c) a number, 2 decimals
answer_25_2d = ...  # d) a number, 2 decimals
answer_25_2e = ...  # e) a number, 2 decimals
answer_25_2f = ...  # f) a whole number
answer_25_2g = ...  # g) a number, 1 decimal
answer_25_2h = ...  # h) a number, 1 decimal

for letter, answer in zip("abcdefgh", [answer_25_2a, answer_25_2b, answer_25_2c, answer_25_2d, answer_25_2e, answer_25_2f, answer_25_2g, answer_25_2h]):
    wb.check(f"25.2{letter}", answer)

**Ex 25.3 — Suivre les formes dans l'autoencodeur convolutif**

In [ ]:
# Ex 25.3: the values you found on paper
answer_25_3a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3d = ...  # d) a whole number
answer_25_3e = ...  # e) a whole number
answer_25_3f = ...  # f) a whole number
answer_25_3g = ...  # g) a number, 2 decimals
answer_25_3h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_25_3a, answer_25_3b, answer_25_3c, answer_25_3d, answer_25_3e, answer_25_3f, answer_25_3g, answer_25_3h]):
    wb.check(f"25.3{letter}", answer)

**Ex 25.4 — Retrouver 28×28 : padding et output_padding d'une convolution transposée**

In [ ]:
# Ex 25.4: the values you found on paper
answer_25_4a = ...  # a) a whole number
answer_25_4b = ...  # b) a whole number
answer_25_4c = ...  # c) a whole number
answer_25_4d = ...  # d) a whole number
answer_25_4e = ...  # e) a whole number
answer_25_4f = ...  # f) a whole number
answer_25_4g = ...  # g) a whole number
answer_25_4h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_25_4a, answer_25_4b, answer_25_4c, answer_25_4d, answer_25_4e, answer_25_4f, answer_25_4g, answer_25_4h]):
    wb.check(f"25.4{letter}", answer)

**Ex 25.6 — Où tombent les tirages d'une gaussienne en grande dimension**

In [ ]:
# Ex 25.6: the values you found on paper
answer_25_6a = ...  # a) a whole number
answer_25_6b = ...  # b) a number, 2 decimals
answer_25_6c = ...  # c) a number, 3 decimals
answer_25_6d = ...  # d) a number, 3 decimals
answer_25_6e = ...  # e) a number, 2 decimals
answer_25_6f = ...  # f) a whole number
answer_25_6g = ...  # g) a number, 2 decimals
answer_25_6h = ...  # h) a number, 2 decimals

for letter, answer in zip("abcdefgh", [answer_25_6a, answer_25_6b, answer_25_6c, answer_25_6d, answer_25_6e, answer_25_6f, answer_25_6g, answer_25_6h]):
    wb.check(f"25.6{letter}", answer)

**Ex 25.7 — La divergence KL entre N(μ, σ²) et N(0, 1)**

In [ ]:
# Ex 25.7: the values you found on paper
answer_25_7a = ...  # a) a number, 3 decimals
answer_25_7b = ...  # b) a number, 3 decimals
answer_25_7c = ...  # c) a number, 3 decimals
answer_25_7d = ...  # d) a number, 3 decimals
answer_25_7e = ...  # e) a number, 3 decimals
answer_25_7f = ...  # f) a number, 3 decimals
answer_25_7g = ...  # g) a number, 3 decimals
answer_25_7h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_25_7a, answer_25_7b, answer_25_7c, answer_25_7d, answer_25_7e, answer_25_7f, answer_25_7g, answer_25_7h]):
    wb.check(f"25.7{letter}", answer)

## Partie A · Autoencodeurs denses

Tu fais apprendre une seule image à un autoencodeur (25.14), tu entraînes le premier autoencodeur du livre puis sa version profonde (25.15, 25.16), tu lui montres des vêtements (25.17), tu dessines son espace latent en 2D (25.18), tu le secoues (25.19), tu interpoles entre deux codes avec ta librairie (25.20, 25.21), tu fais varier le goulot (25.22) et tu compares l'autoencodeur linéaire à l'ACP (25.23).

La cellule d'outils sert aux parties A à C. Elle importe PyTorch, choisit le device et charge MNIST et Fashion-MNIST en lignes de 784 pixels dans $[0, 1]$ (`float32`) : `X_MNIST`, `Y_MNIST`, `X_FASHION`, `Y_FASHION` (les `N_TRAIN` premières images d'un ordre fixé : 10 000 en mode rapide, 60 000 sinon ; les 10 000 premières, `N_FIXED`, sont les mêmes dans les deux modes), et 2 000 images de test de chacun (`XM_TEST`, `YM_TEST`, `XF_TEST`, `YF_TEST`). Elle donne la classe `AutoEncoder` (un encodeur puis un décodeur ; tous les autoencodeurs du notebook lisent et rendent des lignes de 784 pixels dans $[0, 1]$), et des outils qui travaillent sans graphe, en mode évaluation, et rendent des tableaux NumPy : `encode`, `decode`, `reconstruct`, `mse_per_image` (la MSE de chaque image) et `show_rows` (des images en lignes).

In [ ]:
# Tools for the notebook exercises (parts A to C)
import copy
import math
import re
import tempfile
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import subspace_angles
from sklearn.decomposition import PCA
from wb.nbtools import (StopCheck, as_numpy, checked, filled, fr, refuses, returned, run_pytest_group, same,
                        shape_text, verdict, written)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:                       # a computer without PyTorch (Mac Intel): this chapter is done on Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : ce chapitre se fait sur Colab (00_setup/INSTALL_LOCAL.md). Les "
          "cellules qui utilisent PyTorch affichent ⏳ au lieu de s'exécuter ; les tests de ta librairie tournent "
          "sans lui : python -m pytest tests/test_ch25_vae.py -q")

    def _skip_torch_cells(lines):
        """IPython input transformer: without PyTorch, a cell that uses it prints ⏳ instead of stopping "Run all"."""
        if re.search(r"\b(torch|torchvision)\b|(?<![\w.])(nn|F)\.", "".join(lines)):
            return ['print("⏳ PyTorch absent sur cet ordinateur : cette cellule se fait sur Colab.")\n']
        return lines

    try:
        _shell = get_ipython()
    except NameError:                     # not in a notebook
        _shell = None
    if _shell is not None and all(getattr(f, "__name__", "") != "_skip_torch_cells" for f in _shell.input_transformers_cleanup):
        _shell.input_transformers_cleanup.append(_skip_torch_cells)

DEVICE = cfg.device                        # "cuda" if a GPU is available, else "cpu" (wb.setup)
TEST_VAE = "tests/test_ch25_vae.py"        # the tests of your library vae.py
WORK_DIR = Path(tempfile.mkdtemp(prefix="wb_ch25_"))   # a temporary folder for the files the exercises write
FASHION_CLASSES = wb.datasets.FASHION_CLASSES
N_TRAIN = 10_000 if FAST_MODE else 60_000  # training images of the long experiments
N_FIXED = 10_000                           # the predictions (🔮) use these first 10 000 images, in both modes

# MNIST and Fashion-MNIST as float32 rows of 784 pixels in [0, 1]: a fixed order, so that the first N_FIXED
# training images are the same in both modes; 2 000 test images of each
_XM, _YM = wb.datasets.load_mnist("train", flatten=True, normalize=True, seed=None)
_XF, _YF = wb.datasets.load_fashion_mnist("train", flatten=True, normalize=True, seed=None)
_ORDER = np.random.default_rng(2500).permutation(60_000)
X_MNIST, Y_MNIST = _XM[_ORDER[:N_TRAIN]], _YM[_ORDER[:N_TRAIN]]
X_FASHION, Y_FASHION = _XF[_ORDER[:N_TRAIN]], _YF[_ORDER[:N_TRAIN]]
XM_TEST, YM_TEST = wb.datasets.load_mnist("test", n=2000, flatten=True, normalize=True, seed=0)
XF_TEST, YF_TEST = wb.datasets.load_fashion_mnist("test", n=2000, flatten=True, normalize=True, seed=0)
del _XM, _YM, _XF, _YF


def to_tensor(x):
    """A float32 torch tensor (on the CPU) from an array or a tensor."""
    return torch.as_tensor(np.asarray(as_numpy(x), dtype=np.float32))


def n_params(model):
    """The number of parameters (learned numbers) of a torch module."""
    return sum(p.numel() for p in model.parameters())


if torch is not None:
    class AutoEncoder(nn.Module):
        """An encoder and a decoder, one after the other: forward(x) = decoder(encoder(x)).
        Every autoencoder of this notebook reads images as rows of 784 pixels in [0, 1], and returns rows of 784
        pixels in [0, 1] (a convolutional one unflattens and flattens inside)."""

        def __init__(self, encoder, decoder):
            super().__init__()
            self.encoder = encoder
            self.decoder = decoder

        def forward(self, x):
            return self.decoder(self.encoder(x))


def _batched(fn, x, batch_size=1000):
    """fn applied by batches to the rows of x, in eval mode and without a graph: a NumPy float32 array."""
    x = to_tensor(x)
    out = []
    with torch.inference_mode():
        for i in range(0, len(x), batch_size):
            out.append(fn(x[i:i + batch_size].to(DEVICE)).float().cpu())
    return torch.cat(out).numpy()


def encode(model, x, batch_size=1000):
    """The codes model.encoder(x) of the images x (N, 784), as a NumPy array (eval mode, no graph)."""
    model.eval()
    return _batched(model.encoder, x, batch_size)


def decode(model, z, batch_size=1000):
    """The images model.decoder(z) of the codes z, as a NumPy array (N, 784) (eval mode, no graph)."""
    model.eval()
    return _batched(model.decoder, z, batch_size).reshape(len(z), -1)


def reconstruct(model, x, batch_size=1000):
    """The reconstructions model(x) of the images x (N, 784), as a NumPy array (N, 784)."""
    model.eval()
    return _batched(model, x, batch_size).reshape(len(x), -1)


def mse_per_image(a, b):
    """The mean squared error of each image: a NumPy array of N values (a and b: N images of 784 pixels)."""
    a, b = np.asarray(as_numpy(a), dtype=np.float64), np.asarray(as_numpy(b), dtype=np.float64)
    return ((a.reshape(len(a), -1) - b.reshape(len(b), -1)) ** 2).mean(axis=1)


def show_rows(rows, row_titles=None, title=None, size=1.0):
    """Draw images in rows: `rows` is a list of arrays of images (k, 784) or (k, 28, 28), one per row."""
    rows = [np.asarray(as_numpy(r), dtype=np.float32).reshape(-1, 28, 28) for r in rows]
    n_cols = max(len(r) for r in rows)
    fig, axes = plt.subplots(len(rows), n_cols, figsize=(size * n_cols, size * len(rows) + (0.4 if title else 0)),
                             squeeze=False)
    for i, row in enumerate(rows):
        for j in range(n_cols):
            ax = axes[i, j]
            ax.axis("off")
            if j < len(row):
                ax.imshow(row[j], cmap="gray", vmin=0, vmax=1)
        if row_titles:
            axes[i, 0].text(-3, 14, row_titles[i], ha="right", va="center", fontsize=8)
    if title:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


print(f"tools ready · torch {torch.__version__ if torch is not None else 'absent'} · device {DEVICE} · "
      f"MNIST {X_MNIST.shape} · Fashion-MNIST {X_FASHION.shape} · {len(XM_TEST)} + {len(XF_TEST)} test images")

### Ex 25.14 — Prédire : un autoencodeur entraîné sur une seule image 🔮 ★ ⏱️ 15 min
**Objectif :** prédire ce qu'un autoencodeur qui n'a vu qu'une image fait d'autres entrées, et où il range cette image.  
**Prérequis :** Ex 25.1 · ch. 23 (boucle d'entraînement) · fiche §25.3 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

La fiche raconte l'expérience du tigre : un autoencodeur entraîné longtemps sur une seule photo la reconstruit parfaitement, et le livre explique qu'il l'a mémorisée, au point de redonner le tigre à partir d'une image noire. Tu refais l'expérience en petit sur **une** botte de Fashion-MNIST : un autoencodeur 784-20-784 (`train_one_image_14`, 500 pas d'Adam, une graine fixée), puis le même **sans aucun biais** (`bias=False` dans ses deux couches).

**Avant d'exécuter**, écris tes trois prédictions dans la cellule suivante (`True` ou `False`) :
a) l'image **noire**, donnée à l'autoencodeur qui a appris la botte, ressort-elle comme la botte (MSE avec la botte < 0,01) ?
b) le **T-shirt**, donné au même autoencodeur, ressort-il plus proche de la botte que de lui-même (MSE avec la botte plus petite qu'avec le T-shirt) ?
c) **sans aucun biais**, l'autoencodeur reconstruit-il encore la botte (MSE < 0,001) ?

La cellule d'expérience n'entraîne rien tant que tes trois prédictions ne sont pas écrites (deux secondes ensuite). Elle affiche aussi ce que rend le décodeur seul pour un code nul : la part de l'image que portent ses biais.

In [ ]:
BOOT_14 = XF_TEST[np.flatnonzero(YF_TEST == 9)[0]]       # an ankle boot (class 9), 784 pixels in [0, 1]
SHIRT_14 = XF_TEST[np.flatnonzero(YF_TEST == 0)[0]]      # a T-shirt (class 0)
BLACK_14 = np.zeros(784, dtype=np.float32)               # an all-black image


def train_one_image_14(image, n_latent=20, bias=True, steps=500, seed=2514):
    """A 784-n_latent-784 autoencoder (Linear + ReLU, then Linear + Sigmoid), trained on ONE image only:
    `steps` steps of Adam (learning rate 0.001) on the binary cross-entropy. With bias=False, no layer has biases."""
    torch.manual_seed(seed)
    model = AutoEncoder(nn.Sequential(nn.Linear(784, n_latent, bias=bias), nn.ReLU()),
                        nn.Sequential(nn.Linear(n_latent, 784, bias=bias), nn.Sigmoid()))
    x = to_tensor(image).reshape(1, 784)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(steps):
        loss = F.binary_cross_entropy(model(x), x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


show_rows([np.stack([BOOT_14, SHIRT_14, BLACK_14])], ["les trois images"], size=1.3)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
black_14 = ...     # a) the black image, through the autoencoder trained on the boot: does it come out as the
                   #    boot (MSE with the boot < 0.01)? True or False
shirt_14 = ...     # b) the T-shirt, through the same autoencoder: does it come out closer to the boot than to
                   #    itself (smaller MSE with the boot than with the T-shirt)? True or False
no_bias_14 = ...   # c) the same autoencoder WITHOUT any bias (bias=False), trained on the boot: is the boot still
                   #    reconstructed (MSE < 0.001)? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [black_14, shirt_14, no_bias_14]):
    print("⏳ Ex 25.14 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    with checked("25.14"):
        models_14 = {"avec biais": train_one_image_14(BOOT_14), "sans biais": train_one_image_14(BOOT_14, bias=False)}
        print(f"{'':12s} {'botte → botte':>14s} {'noir → botte':>13s} {'T-shirt → botte':>16s} {'T-shirt → T-shirt':>18s} "
              f"{'code nul → botte':>17s}")
        rows_14, titles_14 = [], []
        for name_14, model_14 in models_14.items():
            out_14 = reconstruct(model_14, np.stack([BOOT_14, BLACK_14, SHIRT_14]))
            zero_code_14 = decode(model_14, np.zeros((1, 20), dtype=np.float32))[0]   # the decoder alone, code = 0
            print(f"{name_14:12s} {mse_per_image(out_14[:1], BOOT_14[None])[0]:14.5f} "
                  f"{mse_per_image(out_14[1:2], BOOT_14[None])[0]:13.4f} {mse_per_image(out_14[2:], BOOT_14[None])[0]:16.4f} "
                  f"{mse_per_image(out_14[2:], SHIRT_14[None])[0]:18.4f} {mse_per_image(zero_code_14[None], BOOT_14[None])[0]:17.4f}")
            rows_14.append(np.vstack([out_14, zero_code_14[None]]))
            titles_14.append(name_14)
        show_rows(rows_14, titles_14, "sorties : botte, image noire, T-shirt, puis le décodeur seul avec un code nul", size=1.3)
    wb.check("25.14a", black_14)
    wb.check("25.14b", shirt_14)
    wb.check("25.14c", no_bias_14)

Dans tes notes : lesquelles de tes prédictions étaient justes ? Pourquoi le T-shirt donne-t-il la botte alors que l'image noire ne la donne pas (regarde ce que la première couche fait d'une entrée nulle, puis la colonne « code nul ») ? Où ce réseau a-t-il rangé la botte : dans les biais de sa sortie, ou dans les poids de son décodeur ? Qu'est-ce qui, chez le livre, a pu ranger le tigre dans les biais ?

### Ex 25.15 — Premier autoencodeur dense 784-20-784 en PyTorch 📦 ★★ ⏱️ 25 min
**Objectif :** écrire le premier autoencodeur du livre et sa boucle d'entraînement, dont la cible est l'entrée elle-même.  
**Prérequis :** Ex 25.1 · ch. 23 (Ex 23.18, boucle d'entraînement) · fiche §25.3, §25.4 · **Fil rouge :** MNIST · **Parcours :** R, M, C

Le premier autoencodeur du livre sur MNIST : une couche dense de 784 vers 20 neurones avec une ReLU (l'encodeur), une couche de 20 vers 784 avec une sigmoïde (le décodeur), et une loss qui compare la sortie à l'**entrée**.

**Contrat**
- `make_dense_ae_15(n_latent=20)` rend un `AutoEncoder` dont l'encodeur est `nn.Sequential(nn.Linear(784, n_latent), nn.ReLU())` et le décodeur `nn.Sequential(nn.Linear(n_latent, 784), nn.Sigmoid())` ;
- `fit_ae_15(model, x, epochs, lr=1e-3, batch_size=128, seed=0)` entraîne `model` sur place : `x` est un tableau NumPy `float32` `(N, 784)` dans $[0, 1]$, l'optimiseur `Adam(lr)`, la loss la cross-entropy binaire **moyenne** entre `model(batch)` et le batch lui-même ; les mini-batches suivent un ordre tiré à chaque epoch par `torch.randperm(N, generator=g)`, avec **un seul** `g = torch.Generator().manual_seed(seed)` créé avant la première epoch ; les batches vont sur le device du modèle ; la fonction rend la liste des losses moyennes de chaque epoch (chaque batch pesé par sa taille).

La vérification regarde l'architecture et la sortie, puis entraîne deux fois trois epochs sur 1 024 images (la même graine doit donner les mêmes losses). La cellule suivante entraîne trois autoencodeurs, à 20, 10 et 3 latents (une dizaine de secondes en mode rapide), et affiche leurs reconstructions.

Dans tes notes : comment la qualité change-t-elle de 20 à 3 latents, et retrouves-tu ce que décrit la fiche (un 4 qui tourne au 9, des taches) ? Pourquoi une sigmoïde en sortie, et pourquoi la cross-entropy binaire convient-elle à des pixels qui ne sont pas des 0 et des 1 ?

In [ ]:
def make_dense_ae_15(n_latent=20):
    """The book's first autoencoder: an AutoEncoder whose encoder is nn.Sequential(nn.Linear(784, n_latent),
    nn.ReLU()) and whose decoder is nn.Sequential(nn.Linear(n_latent, 784), nn.Sigmoid())."""
    raise NotImplementedError("make_dense_ae_15")


def fit_ae_15(model, x, epochs, lr=1e-3, batch_size=128, seed=0):
    """Train `model` in place on the images x (a float32 NumPy array (N, 784) in [0, 1]) to give back its input:
    Adam(lr), the mean binary cross-entropy between model(batch) and the batch, mini-batches of `batch_size` in an
    order drawn each epoch by torch.randperm(N, generator=g), with ONE g = torch.Generator().manual_seed(seed) made
    before the first epoch. Returns the list of the mean losses of each epoch (each batch weighted by its size)."""
    raise NotImplementedError("fit_ae_15")

In [ ]:
with checked("25.15"):
    torch.manual_seed(0)
    model_15 = make_dense_ae_15(20)
    if returned("25.15", "make_dense_ae_15", model_15):
        layers_15 = [type(m).__name__ for m in model_15.encoder] + ["|"] + [type(m).__name__ for m in model_15.decoder]
        verdict("25.15", isinstance(model_15, AutoEncoder) and layers_15 == ["Linear", "ReLU", "|", "Linear", "Sigmoid"],
                "make_dense_ae_15 : un AutoEncoder (Linear, ReLU | Linear, Sigmoid).",
                f"attendu AutoEncoder(Sequential(Linear, ReLU), Sequential(Linear, Sigmoid)), reçu {layers_15}.")
        shapes_15 = [(m.in_features, m.out_features) for m in model_15.modules() if isinstance(m, nn.Linear)]
        verdict("25.15", shapes_15 == [(784, 20), (20, 784)], "les deux couches font 784 → 20 et 20 → 784.",
                f"tailles des couches : attendu [(784, 20), (20, 784)], reçu {shapes_15}.")
        out_15 = reconstruct(model_15, X_MNIST[:5])
        verdict("25.15", out_15.shape == (5, 784) and out_15.min() >= 0 and out_15.max() <= 1,
                "une sortie (5, 784) dans [0, 1].", f"la sortie doit être (5, 784) dans [0, 1], reçu {out_15.shape}.")
    class _Spy15(nn.Module):                     # a fake model that remembers what it receives and returns
        def __init__(self):
            super().__init__()
            self.w = nn.Parameter(torch.zeros(1))
            self.seen = []

        def forward(self, x):
            out = torch.sigmoid(self.w + 0 * x)
            self.seen.append((x.detach().cpu().clone(), out.detach().cpu().clone()))
            return out
    x_spy_15 = np.linspace(0.05, 0.95, 10 * 784, dtype=np.float32).reshape(10, 784)   # 10 distinct images
    spy_15 = _Spy15()
    spy_losses_15 = fit_ae_15(spy_15, x_spy_15, 2, batch_size=4, seed=3)
    if returned("25.15", "fit_ae_15", spy_losses_15):
        firsts_15 = [int(np.argmin(np.abs(x_spy_15[:, 0] - float(v)))) for xb, _ in spy_15.seen for v in xb[:, 0]]
        g_15 = torch.Generator().manual_seed(3)
        want_order_15 = torch.randperm(10, generator=g_15).tolist() + torch.randperm(10, generator=g_15).tolist()
        verdict("25.15", firsts_15 == want_order_15,
                "les mini-batches suivent l'ordre d'un seul générateur, tiré à chaque epoch.",
                f"ordre des images vues : attendu {want_order_15}, reçu {firsts_15} (un seul torch.Generator créé "
                "avant les epochs, un torch.randperm par epoch).")
        want_losses_15 = []
        for epoch_15 in range(2):
            batches_15 = spy_15.seen[3 * epoch_15:3 * epoch_15 + 3]
            want_losses_15.append(sum(F.binary_cross_entropy(o, xb).item() * len(xb) for xb, o in batches_15) / 10)
        same("25.15", "les deux losses moyennes du faux modèle (cross-entropy binaire face à l'entrée, chaque batch "
             "pesé par sa taille)", spy_losses_15, want_losses_15, rtol=1e-5)
    torch.manual_seed(0)
    losses_15 = fit_ae_15(make_dense_ae_15(20), X_MNIST[:1024], epochs=3, seed=1)
    if returned("25.15", "fit_ae_15", losses_15):
        torch.manual_seed(0)
        again_15 = fit_ae_15(make_dense_ae_15(20), X_MNIST[:1024], epochs=3, seed=1)
        verdict("25.15", len(losses_15) == 3 and np.allclose(losses_15, again_15, rtol=1e-5)
                and losses_15[2] < losses_15[0] < 0.75,
                f"fit_ae_15 : trois losses qui baissent ({', '.join(fr(v, 3) for v in losses_15)}), les mêmes à chaque appel.",
                f"attendu une loss par epoch, qui baisse et soit la même à chaque appel, reçu {losses_15!r:.200} puis "
                f"{again_15!r:.200} (un seul générateur, créé avant la première epoch ; la cible est l'entrée).")

In [ ]:
with checked("25.15"):
    EPOCHS_15 = 5 if FAST_MODE else 20
    results_15 = {}
    for n_latent_15 in (20, 10, 3):
        torch.manual_seed(2515)
        model_15 = make_dense_ae_15(n_latent_15).to(DEVICE)
        start_15 = time.perf_counter()
        fit_ae_15(model_15, X_MNIST, EPOCHS_15, seed=2515)
        results_15[n_latent_15] = (reconstruct(model_15, XM_TEST), time.perf_counter() - start_15)
        print(f"{n_latent_15:2d} latents : MSE de test {mse_per_image(results_15[n_latent_15][0], XM_TEST).mean():.4f} "
              f"({results_15[n_latent_15][1]:.1f} s)")
    show_rows([XM_TEST[:10]] + [results_15[n][0][:10] for n in (20, 10, 3)],
              ["originaux", "20 latents", "10 latents", "3 latents"], f"{len(X_MNIST)} images, {EPOCHS_15} epochs")

La cellule suivante donne les versions de référence de `make_dense_ae` et de la boucle, `train_ae`, qui rend aussi un historique : les exercices suivants s'en servent.

In [ ]:
# From here on: the reference versions of make_dense_ae and of the training loop (your 25.15, plus a history)
def make_dense_ae(n_latent=20):
    """The autoencoder of 25.15: Linear(784, n_latent) + ReLU, then Linear(n_latent, 784) + Sigmoid."""
    return AutoEncoder(nn.Sequential(nn.Linear(784, n_latent), nn.ReLU()),
                       nn.Sequential(nn.Linear(n_latent, 784), nn.Sigmoid()))


def train_ae(model, x, epochs, lr=1e-3, batch_size=128, seed=0, loss="bce", x_val=None, verbose=False):
    """Train the autoencoder `model` in place (on DEVICE) to give back the images x (N, 784): Adam(lr), mini-batches
    in an order drawn from `seed`, the mean binary cross-entropy (loss="bce") or squared error (loss="mse").
    Returns the history: "loss" (mean of each epoch), "val_mse" (MSE of x_val after each epoch, if given),
    "seconds" (duration of each epoch)."""
    model.to(DEVICE)
    x = to_tensor(x)
    loss_fn = F.binary_cross_entropy if loss == "bce" else F.mse_loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    history = {"loss": [], "val_mse": [], "seconds": []}
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        total = 0.0
        order = torch.randperm(len(x), generator=g)
        for i in range(0, len(x), batch_size):
            xb = x[order[i:i + batch_size]].to(DEVICE)
            batch_loss = loss_fn(model(xb), xb)
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()
            total += batch_loss.item() * len(xb)
        history["loss"].append(total / len(x))
        history["seconds"].append(time.perf_counter() - start)
        if x_val is not None:
            history["val_mse"].append(float(mse_per_image(reconstruct(model, x_val), x_val).mean()))
        if verbose:
            print(f"epoch {epoch:2d} · loss {history['loss'][-1]:.4f}" +
                  (f" · val MSE {history['val_mse'][-1]:.4f}" if x_val is not None else ""))
    return history

### Ex 25.16 — Autoencodeur profond 512-256-128-20 📦 ★★ ⏱️ 25 min
**Objectif :** construire un autoencodeur profond symétrique et mesurer ce que la profondeur apporte à goulot égal.  
**Prérequis :** Ex 25.15 · fiche §25.4 · **Fil rouge :** MNIST · **Parcours :** R, M, C

Le livre passe à un autoencodeur **profond** : trois couches cachées de 512, 256 et 128 neurones avant le code, et le chemin inverse après. Ton code n'a pas d'activation : ses valeurs peuvent être négatives (le livre mettait une ReLU au goulot, d'où ses latents tous positifs).

**Contrat** : `make_deep_ae_16(n_latent=20, hidden=(512, 256, 128))` rend un `AutoEncoder` dont l'encodeur est `Linear(784, hidden[0])` + ReLU, `Linear(hidden[0], hidden[1])` + ReLU…, puis `Linear(hidden[-1], n_latent)` **sans** activation, et le décodeur le miroir : `Linear(n_latent, hidden[-1])` + ReLU…, `Linear(hidden[0], 784)` + sigmoïde ; encodeur et décodeur sont des `nn.Sequential`.

La vérification compare la suite des couches à celle attendue, pour deux réglages. La cellule suivante entraîne le dense de 25.15 et ton profond, avec le même `train_ae` (8 epochs en mode rapide, une quinzaine de secondes), et compare leurs erreurs de test.

Dans tes notes : combien de paramètres le profond a-t-il de plus, et que gagne-t-il ? Avec la fiche (« assez de latents et assez de capacité »), qu'est-ce qui limite chacun des deux modèles ?

In [ ]:
def make_deep_ae_16(n_latent=20, hidden=(512, 256, 128)):
    """A deep autoencoder: the encoder is Linear(784, hidden[0]) + ReLU, Linear(hidden[0], hidden[1]) + ReLU, ...,
    then Linear(hidden[-1], n_latent) WITHOUT activation (the code can be negative); the decoder is the mirror
    image: Linear(n_latent, hidden[-1]) + ReLU, ..., Linear(hidden[0], 784) + Sigmoid. Returns an AutoEncoder
    whose encoder and decoder are nn.Sequential."""
    raise NotImplementedError("make_deep_ae_16")

In [ ]:
with checked("25.16"):
    torch.manual_seed(0)
    model_16 = make_deep_ae_16(20)
    if returned("25.16", "make_deep_ae_16", model_16):
        def _layout_16(seq):
            return [(m.in_features, m.out_features) if isinstance(m, nn.Linear) else type(m).__name__ for m in seq]
        want_enc_16 = [(784, 512), "ReLU", (512, 256), "ReLU", (256, 128), "ReLU", (128, 20)]
        want_dec_16 = [(20, 128), "ReLU", (128, 256), "ReLU", (256, 512), "ReLU", (512, 784), "Sigmoid"]
        verdict("25.16", _layout_16(model_16.encoder) == want_enc_16, "l'encodeur : 784 → 512 → 256 → 128 → 20, sans activation sur le code.",
                f"encodeur attendu {want_enc_16}, reçu {_layout_16(model_16.encoder)}.")
        verdict("25.16", _layout_16(model_16.decoder) == want_dec_16, "le décodeur : son miroir, terminé par une sigmoïde.",
                f"décodeur attendu {want_dec_16}, reçu {_layout_16(model_16.decoder)}.")
        other_16 = make_deep_ae_16(2, hidden=(64,))
        verdict("25.16", _layout_16(other_16.encoder) == [(784, 64), "ReLU", (64, 2)]
                and _layout_16(other_16.decoder) == [(2, 64), "ReLU", (64, 784), "Sigmoid"],
                "make_deep_ae_16(2, hidden=(64,)) suit aussi n_latent et hidden.",
                "make_deep_ae_16 doit suivre n_latent et hidden (essayé avec 2 et (64,)).")

In [ ]:
with checked("25.16"):
    EPOCHS_16 = 8 if FAST_MODE else 20
    histories_16 = {}
    for name_16, maker_16 in {"dense 784-20-784": make_dense_ae, "profond 512-256-128-20": make_deep_ae_16}.items():
        torch.manual_seed(2516)
        model_16 = maker_16(20)
        histories_16[name_16] = (model_16, train_ae(model_16, X_MNIST, EPOCHS_16, seed=2516, x_val=XM_TEST))
        print(f"{name_16:24s} {n_params(model_16):9,d} paramètres · MSE de test {histories_16[name_16][1]['val_mse'][-1]:.4f} "
              f"· {sum(histories_16[name_16][1]['seconds']):.1f} s")
    mse_dense_16, mse_deep_16 = (h["val_mse"][-1] for _, h in histories_16.values())
    verdict("25.16", mse_deep_16 < mse_dense_16, "à 20 latents, le profond reconstruit mieux que le dense.",
            "le profond devrait battre le dense à 20 latents : vérifie ton architecture.")
    show_rows([XM_TEST[:10]] + [reconstruct(m, XM_TEST[:10]) for m, _ in histories_16.values()],
              ["originaux", "dense", "profond"], f"{EPOCHS_16} epochs sur {len(X_MNIST)} images")

In [ ]:
# The reference deep autoencoder (your 25.16), and AE_DEEP: the one that parts A and C explore (25.17 to 25.21,
# 25.31), trained the same way in both modes (the first N_FIXED images, 8 epochs, seed 2516; about 10 s on a CPU)
def make_deep_ae(n_latent=20, hidden=(512, 256, 128)):
    """The deep autoencoder of 25.16: dense ReLU layers, a linear code, a sigmoid output."""
    sizes = [784, *hidden, n_latent]
    encoder, decoder = [], []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        encoder += [nn.Linear(n_in, n_out), nn.ReLU()]
    for n_in, n_out in zip(sizes[::-1][:-1], sizes[::-1][1:]):
        decoder += [nn.Linear(n_in, n_out), nn.ReLU()]
    return AutoEncoder(nn.Sequential(*encoder[:-1]), nn.Sequential(*decoder[:-1], nn.Sigmoid()))


torch.manual_seed(2516)
AE_DEEP = make_deep_ae(20)
train_ae(AE_DEEP, X_MNIST[:N_FIXED], 8, seed=2516)
print(f"AE_DEEP : MSE de test {mse_per_image(reconstruct(AE_DEEP, XM_TEST), XM_TEST).mean():.4f}")

### Ex 25.17 — Prédire : ce qu'un autoencodeur de chiffres fait d'un vêtement 🔮 ★★ ⏱️ 15 min
**Objectif :** prédire, puis mesurer, l'erreur d'un autoencodeur sur des entrées d'un autre domaine que ses données.  
**Prérequis :** Ex 25.16 · 25.Q11 · fiche §25.5.4, §25.7.2 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

La fiche le dit : un autoencodeur de chiffres ne sait redessiner que des morceaux de chiffres. Mais de combien se trompe-t-il, et sur quels vêtements ? `AE_DEEP` (l'autoencodeur profond à 20 latents de la cellule précédente, entraîné sur des chiffres seulement) reçoit les 2 000 vêtements de test.

**Avant d'exécuter**, écris tes prédictions :
a) la MSE moyenne des vêtements divisée par celle des chiffres de test : `"A"` moins de 2, `"B"` entre 2 et 5, `"C"` plus de 5 ;
b) les **deux** classes de vêtements les mieux reconstruites (MSE moyennes les plus petites) : `"A"` Trouser et Dress (pantalon et robe), `"B"` Sandal et Sneaker (sandale et basket), `"C"` Bag et Ankle boot (sac et bottine), `"D"` T-shirt/top et Pullover ;
c) la MSE moyenne des vêtements est-elle plus petite que celle d'une réponse toute noire (0 partout) ? (`True` ou `False`)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
ratio_17 = ...    # a) mean MSE of the clothes divided by the mean MSE of the digits: "A" less than 2,
                  #    "B" between 2 and 5, "C" more than 5
best_17 = ...     # b) the two best reconstructed classes of clothes (smallest mean MSE): "A" Trouser and Dress,
                  #    "B" Sandal and Sneaker, "C" Bag and Ankle boot, "D" T-shirt/top and Pullover
black_17 = ...    # c) is the mean MSE of the clothes smaller than that of an all-black guess (0 everywhere)?
                  #    True or False

In [ ]:
if any(answer is ... or answer is None for answer in [ratio_17, best_17, black_17]):
    print("⏳ Ex 25.17 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    with checked("25.17"):
        mse_digits_17 = mse_per_image(reconstruct(AE_DEEP, XM_TEST), XM_TEST)
        out_clothes_17 = reconstruct(AE_DEEP, XF_TEST)
        mse_clothes_17 = mse_per_image(out_clothes_17, XF_TEST)
        mse_black_17 = mse_per_image(np.zeros_like(XF_TEST), XF_TEST)
        print(f"MSE moyenne · chiffres {mse_digits_17.mean():.4f} · vêtements {mse_clothes_17.mean():.4f} "
              f"(rapport {mse_clothes_17.mean() / mse_digits_17.mean():.2f}) · image noire sur les vêtements {mse_black_17.mean():.4f}")
        for c_17 in np.argsort([mse_clothes_17[YF_TEST == c].mean() for c in range(10)]):
            print(f"  {FASHION_CLASSES[c_17]:12s} {mse_clothes_17[YF_TEST == c_17].mean():.4f}")
        first_17 = [np.flatnonzero(YF_TEST == c)[0] for c in range(10)]
        show_rows([XF_TEST[first_17], out_clothes_17[first_17]], ["vêtements", "reconstruits"],
                  "un vêtement de chaque classe, à travers l'autoencodeur de chiffres", size=1.1)
    wb.check("25.17a", ratio_17)
    wb.check("25.17b", best_17)
    wb.check("25.17c", black_17)

Dans tes notes : qu'ont en commun les vêtements les mieux reconstruits, et à quels chiffres ressemblent leurs reconstructions ? Que t'apprend c) sur ce que l'autoencodeur a quand même appris des images en général ? Quel usage pratique tire-t-on de cet écart d'erreur (25.37) ?

### Ex 25.18 — Reproduire la carte latente 2D et sa grille décodée (fig. 25.29-25.30) 🎨 ★★ ⏱️ 30 min
**Objectif :** dessiner l'espace latent d'un autoencodeur à 2 latents et décoder une grille de codes.  
**Prérequis :** Ex 25.16 · fiche §25.5.2 · **Fil rouge :** MNIST · **Parcours :** C

Le livre réduit chaque chiffre à **deux** nombres, place les codes de 10 000 chiffres sur un plan (fig. 25.29), puis décode une grille régulière de codes (fig. 25.30). La cellule donnée entraîne `AE_2D`, l'autoencodeur profond de 25.16 avec un code de 2 nombres.

**Contrat**
- `grid_codes_18(lims, n)` rend les $n \times n$ codes d'une grille régulière sur `lims = (xmin, xmax, ymin, ymax)`, en tableau `(n * n, 2)`, ligne par ligne **comme dans une image** : la première ligne a $y$ = `ymax` (le haut), la dernière $y$ = `ymin` ; dans chaque ligne, $x$ va de `xmin` à `xmax` ; les deux bords sont inclus ;
- `plot_latent_map_18(model, x, labels, n=15)` trace deux panneaux côte à côte : à gauche, les codes de `x` en points, colorés par chiffre (`labels`), avec une légende ; à droite, les $n \times n$ images décodées de `grid_codes_18` sur les **mêmes limites** que le panneau de gauche (le min et le max des codes), assemblées en **une** image de $28n \times 28n$ pixels, le code du coin en haut à gauche en premier ; elle finit par `plt.show()` et rend cette mosaïque.

La vérification essaie `grid_codes_18` sur une petite grille, puis trace la carte des 2 000 chiffres de test et compare le coin en haut à gauche de ta mosaïque au décodage de son code.

Dans tes notes : quels chiffres ont des régions bien à eux, lesquels se chevauchent ? Où sont les zones vides, et que décode la grille à ces endroits ? Pourquoi est-ce un problème pour générer des chiffres en tirant des codes au hasard ?

In [ ]:
# AE_2D: a deep autoencoder with a code of 2 numbers, trained on the first N_FIXED images (10 epochs, ≈ 12 s)
torch.manual_seed(2518)
AE_2D = make_deep_ae(2)
train_ae(AE_2D, X_MNIST[:N_FIXED], 10, seed=2518)
print(f"AE_2D : MSE de test {mse_per_image(reconstruct(AE_2D, XM_TEST), XM_TEST).mean():.4f}")

In [ ]:
def grid_codes_18(lims, n):
    """The n × n codes of a regular grid over lims = (xmin, xmax, ymin, ymax), as an array (n * n, 2), row by row
    as in an image: the first row has y = ymax (the top), the last y = ymin; in each row, x goes from xmin to xmax.
    Both ends are included (np.linspace)."""
    raise NotImplementedError("grid_codes_18")


def plot_latent_map_18(model, x, labels, n=15):
    """Two panels side by side: (left) the codes model.encoder(x) as points (x, y), coloured by digit (labels),
    with a legend; (right) the n × n images decoded from grid_codes_18 over the same limits as the left panel
    (the min and max of the codes), assembled in ONE image of (28 n) × (28 n) pixels, the code of the top-left
    corner first. Ends with plt.show(). Returns the mosaic (an array (28 n, 28 n))."""
    raise NotImplementedError("plot_latent_map_18")

In [ ]:
with checked("25.18"):
    grid_18 = grid_codes_18((0.0, 1.0, -2.0, 2.0), 3)
    if returned("25.18", "grid_codes_18", grid_18):
        same("25.18", "grid_codes_18((0, 1, -2, 2), 3)", grid_18,
             [[0, 2], [0.5, 2], [1, 2], [0, 0], [0.5, 0], [1, 0], [0, -2], [0.5, -2], [1, -2]])
    mosaic_18 = plot_latent_map_18(AE_2D, XM_TEST, YM_TEST, n=15)
    if returned("25.18", "plot_latent_map_18", mosaic_18):
        codes_18 = encode(AE_2D, XM_TEST)
        corner_18 = decode(AE_2D, np.array([[codes_18[:, 0].min(), codes_18[:, 1].max()]], dtype=np.float32))
        verdict("25.18", np.shape(mosaic_18) == (420, 420) and np.allclose(np.asarray(mosaic_18)[:28, :28],
                                                                           corner_18.reshape(28, 28), atol=1e-5),
                "la mosaïque (420, 420) commence par le code du coin en haut à gauche (x min, y max).",
                f"attendu une mosaïque (420, 420) dont l'image en haut à gauche décode (x min, y max), reçu "
                f"{shape_text(mosaic_18)} (transpose(0, 2, 1, 3) avant le reshape).")

### Ex 25.19 — Secouer les variables latentes : bruit global, une seule composante, latents aléatoires 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer la tolérance du décodeur à un bruit sur les codes, latent par latent, et le décodage de codes au hasard.  
**Prérequis :** Ex 25.16 · fiche §25.5, §25.5.1 · **Fil rouge :** MNIST · **Parcours :** C

Le livre secoue les 20 latents de ses chiffres avec des bruits uniformes de plus en plus forts, puis un seul latent à la fois, puis remplace les codes par des nombres au hasard. Tu refais les trois expériences sur `AE_DEEP` ; comme ses latents n'ont pas l'échelle de ceux du livre, les bruits sont comptés en **écarts-types** des latents.

**Contrat**
- `add_noise_19(codes, scale, rng, column=None)` rend un **nouveau** tableau : `codes` `(N, d)` plus un bruit uniforme dans $[-\text{scale}, \text{scale}]$, tiré par `rng.uniform(-scale, scale, size=...)`, sur toutes les valeurs (`size=codes.shape`) si `column` vaut `None`, sinon sur cette colonne seulement (`size=N`) ; `codes` n'est pas modifié ;
- `noise_mse_19(model, x, scales, seed=0, column=None)` : pour chaque valeur de `scales`, les codes `encode(model, x)`, bruités par `add_noise_19(..., scale, np.random.default_rng(seed), column)` (un **nouveau** générateur pour chaque valeur), puis décodés ; elle rend la liste des MSE moyennes entre ces images et `x`.

La vérification compare tes bruits à ceux d'un générateur de même graine. La cellule suivante trace l'erreur selon le bruit, puis l'erreur quand un seul latent est bruité (un diagramme en barres), et décode des codes tirés au hasard entre le min et le max de chaque latent.

Dans tes notes : à partir de quel bruit les chiffres deviennent-ils illisibles ? Les latents comptent-ils tous autant ? Que donnent les codes au hasard, et pourquoi (25.18) ?

In [ ]:
def add_noise_19(codes, scale, rng, column=None):
    """A new array: codes (N, d) plus uniform noise in [-scale, scale], drawn by rng.uniform(-scale, scale, size=...):
    on every value (size = codes.shape) if column is None, otherwise on that column only (size = N, the other
    columns unchanged). `codes` is not modified."""
    raise NotImplementedError("add_noise_19")


def noise_mse_19(model, x, scales, seed=0, column=None):
    """For each scale of `scales`: the codes encode(model, x), plus add_noise_19(..., scale,
    np.random.default_rng(seed), column) (a NEW generator for each scale), decoded; returns the list of the mean
    MSE between these decoded images and x."""
    raise NotImplementedError("noise_mse_19")

In [ ]:
with checked("25.19"):
    toy_19 = np.arange(6, dtype=np.float32).reshape(3, 2)
    noisy_19 = add_noise_19(toy_19, 0.5, np.random.default_rng(1))
    if returned("25.19", "add_noise_19", noisy_19):
        same("25.19", "add_noise_19(toy, 0.5, default_rng(1)) : tout le tableau",
             noisy_19, toy_19 + np.random.default_rng(1).uniform(-0.5, 0.5, size=(3, 2)), rtol=1e-6, atol=1e-6)
        same("25.19", "add_noise_19(toy, 0.5, default_rng(1), column=1) : la colonne 1 seulement",
             add_noise_19(toy_19, 0.5, np.random.default_rng(1), column=1),
             np.column_stack([toy_19[:, 0], toy_19[:, 1] + np.random.default_rng(1).uniform(-0.5, 0.5, size=3)]),
             rtol=1e-6, atol=1e-6)
        verdict("25.19", np.array_equal(toy_19, np.arange(6).reshape(3, 2)), "codes n'est pas modifié.",
                "add_noise_19 a modifié son entrée : travaille sur une copie.")
    mse_19 = noise_mse_19(AE_DEEP, XM_TEST[:200], [0.0, 1.0])
    if returned("25.19", "noise_mse_19", mse_19):
        clean_19 = mse_per_image(reconstruct(AE_DEEP, XM_TEST[:200]), XM_TEST[:200]).mean()
        verdict("25.19", len(mse_19) == 2 and abs(mse_19[0] - clean_19) < 1e-6 and mse_19[1] > mse_19[0],
                "noise_mse_19 : un bruit nul redonne l'erreur de reconstruction, un bruit de 1 l'augmente.",
                f"attendu [MSE sans bruit = {clean_19:.4f}, une valeur plus grande], reçu {mse_19!r:.200}.")
        twice_19 = noise_mse_19(AE_DEEP, XM_TEST[:200], [1.0, 1.0])
        verdict("25.19", len(twice_19) == 2 and abs(twice_19[0] - twice_19[1]) < 1e-9,
                "deux fois la même échelle donnent la même erreur : un générateur neuf pour chaque échelle.",
                f"noise_mse_19(..., [1.0, 1.0]) doit rendre deux valeurs égales, reçu {twice_19!r:.200} : crée un "
                "nouveau np.random.default_rng(seed) pour chaque échelle.")

In [ ]:
with checked("25.19"):
    codes_19 = encode(AE_DEEP, XM_TEST)
    spread_19 = float(codes_19.std())                      # the typical spread of a latent value
    rel_19 = (0.0, 0.1, 0.3, 1.0, 3.0)
    curve_19 = noise_mse_19(AE_DEEP, XM_TEST, [r * spread_19 for r in rel_19], seed=19)
    print(f"écart-type des latents : {spread_19:.2f} ; leurs valeurs vont de {codes_19.min():.1f} à {codes_19.max():.1f}")
    for r_19, m_19 in zip(rel_19, curve_19):
        print(f"bruit de ± {r_19:.1f} écart-type sur tous les latents : MSE {m_19:.4f}")
    one_19 = [noise_mse_19(AE_DEEP, XM_TEST, [3 * spread_19], seed=19, column=j)[0] for j in range(codes_19.shape[1])]
    fig, ax = plt.subplots(figsize=(8, 2.6))
    ax.bar(range(len(one_19)), one_19)
    ax.axhline(curve_19[0], color="k", lw=1, ls="--", label="sans bruit")
    ax.set_xlabel("latent bruité (± 3 écarts-types, lui seul)")
    ax.set_ylabel("MSE")
    ax.legend()
    plt.show()
    rng_19 = np.random.default_rng(1919)
    random_19 = rng_19.uniform(codes_19.min(axis=0), codes_19.max(axis=0), size=(10, codes_19.shape[1])).astype(np.float32)
    rows_19 = [XM_TEST[:10]] + [decode(AE_DEEP, add_noise_19(codes_19[:10], r * spread_19, np.random.default_rng(19)))
                                for r in (0.3, 1.0, 3.0)] + [decode(AE_DEEP, random_19)]
    show_rows(rows_19, ["originaux", "± 0,3 σ", "± 1 σ", "± 3 σ", "codes au hasard"])

### Ex 25.20 — Interpoler entre deux codes : linéaire et sphérique 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire l'interpolation linéaire et sphérique (slerp) de deux codes dans ta librairie.  
**Prérequis :** Ex 25.6 · fiche §25.5.3 (encadré slerp) · **mylearn :** `vae.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/vae.py` (créé par `python tools/start_chapter.py 25`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur quelques cas ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_ch25_vae.py -q` les lance tous dans un terminal.

**Contrat commun de `vae.py`** (les docstrings, plus ces précisions que vérifient les tests) : les entrées peuvent être des listes et ne sont jamais modifiées ; chaque tableau rendu est un **nouveau** tableau `float64`, jamais une vue d'une entrée (même quand il lui est égal) ; une entrée qui contient un NaN lève `ValueError`.

Écris `interpolate_latents(z0, z1, n_steps, method="linear")` dans `vae.py`. **Contrat** propre à cette fonction :
- `z0` et `z1` sont deux vecteurs 1-D de même forme, sinon `ValueError` ; `n_steps` est un entier ≥ 2 (un `bool` ou un flottant sont refusés), `method` vaut `"linear"` ou `"slerp"`, sinon `ValueError` ;
- les valeurs de $t$ sont `np.linspace(0, 1, n_steps)`, et la fonction rend un tableau `(n_steps, d)` dont la première ligne est **exactement** `z0` et la dernière **exactement** `z1` ;
- `"linear"` : $(1 - t)\,\mathbf{z}_0 + t\,\mathbf{z}_1$ ; `"slerp"` : la formule de la fiche, avec $\Omega$ l'angle entre les deux vecteurs, qui s'applique à des vecteurs de **normes quelconques** ; elle revient à la ligne droite quand une norme est nulle ou quand $\sin\Omega$ est de l'ordre des erreurs d'arrondi (vecteurs parallèles ou opposés), sans émettre de `RuntimeWarning` : teste ces cas **avant** de diviser.

La cellule ci-dessous prend les codes d'un 1 et d'un 0 dans l'espace latent de `AE_DEEP`. La première cellule de vérification essaie ta fonction sur une ligne droite et un quart de cercle, puis décode les deux chemins du 1 au 0 et affiche la norme des codes en route. La seconde lance les tests (`-k "test_interpolate_latents_"`).

Dans tes notes : la norme des codes reste-t-elle constante le long du slerp, et le long de la ligne droite ? Ici, les images des deux chemins se ressemblent-elles ? Dans quel espace latent la différence devrait-elle compter le plus (fiche, et 25.Q9) ?

In [ ]:
# Nothing to write here: interpolate_latents goes in mon_travail/mylearn/vae.py
z0_20 = encode(AE_DEEP, XM_TEST[np.flatnonzero(YM_TEST == 1)[:1]])[0]    # the code of a 1 ...
z1_20 = encode(AE_DEEP, XM_TEST[np.flatnonzero(YM_TEST == 0)[:1]])[0]    # ... and that of a 0

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="25")   # reload your saved files
with checked("25.20"):
    V = mylearn.vae
    a_20, b_20 = np.array([0.0, 0.0]), np.array([1.0, 2.0])
    same("25.20", "interpolate_latents((0, 0), (1, 2), 3) : la ligne droite", V.interpolate_latents(a_20, b_20, 3),
         [[0, 0], [0.5, 1], [1, 2]])
    u_20, w_20 = np.array([2.0, 0.0, 0.0]), np.array([0.0, 2.0, 0.0])
    t_20 = np.linspace(0, 1, 4)[:, None]
    same("25.20", "slerp de (2, 0, 0) à (0, 2, 0) en 4 étapes : l'arc de cercle de rayon 2",
         V.interpolate_latents(u_20, w_20, 4, method="slerp"),
         np.sin((1 - t_20) * np.pi / 2) * u_20 + np.sin(t_20 * np.pi / 2) * w_20)
    path_lin_20 = V.interpolate_latents(z0_20, z1_20, 8)
    path_slerp_20 = V.interpolate_latents(z0_20, z1_20, 8, method="slerp")
    print("normes des codes, ligne droite :", np.round(np.linalg.norm(path_lin_20, axis=1), 1))
    print("normes des codes, slerp        :", np.round(np.linalg.norm(path_slerp_20, axis=1), 1))
    show_rows([decode(AE_DEEP, np.asarray(path_lin_20, dtype=np.float32)),
               decode(AE_DEEP, np.asarray(path_slerp_20, dtype=np.float32))], ["ligne droite", "slerp"],
              "d'un 1 à un 0 dans l'espace latent de AE_DEEP")

In [ ]:
if written(lambda: mylearn.vae.interpolate_latents([0.0], [1.0], 2)):
    run_pytest_group(TEST_VAE, "test_interpolate_latents_")
else:
    print("⏳ Ex 25.20 : les tests attendent tes fonctions.")

### Ex 25.21 — Mélanger dans l'espace des pixels ou dans l'espace latent 🔬 ★★ ⏱️ 25 min
**Objectif :** comparer un fondu de pixels à une interpolation de codes, et mesurer à quel point chaque étape est « un chiffre ».  
**Prérequis :** Ex 25.20 · Ex 25.16 · ✏️ 25.2 · fiche §25.2.3, §25.5.3 · **Fil rouge :** MNIST · **Parcours :** R, C

Le ✏️ 25.2 opposait le mélange de contenu (les pixels) au mélange paramétrique. Un autoencodeur donne des « paramètres » appris : ses codes. Tu compares trois façons de passer d'un chiffre à un autre, en sept étapes : le fondu des pixels, la ligne droite entre les codes, le slerp entre les codes.

**Contrat**
- `blend_pixels_21(x0, x1, n_steps)` rend les images $(1 - t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$ pour $t$ = `np.linspace(0, 1, n_steps)`, un tableau `(n_steps, 784)` ;
- `blend_codes_21(model, x0, x1, n_steps, method="linear")` encode `x0` et `x1`, interpole leurs codes avec **ta** `mylearn.vae.interpolate_latents(..., n_steps, method=method)` et décode : un tableau `(n_steps, 784)` ;
- `off_manifold_21(model, images)` rend, pour chaque image, la MSE entre elle et sa reconstruction par `model` : une mesure de « ce que l'autoencodeur ne sait pas redessiner ».

La vérification essaie tes trois fonctions sur des cas connus. La cellule suivante montre les trois mélanges d'un 1 à un 0, puis entre deux 5 différents, et affiche pour chaque étape la distance à ce que l'autoencodeur sait refaire.

Dans tes notes : à mi-chemin, laquelle des trois images ressemble le plus à un chiffre ? La distance à ce que l'autoencodeur sait refaire sépare-t-elle les fondus de pixels des vrais chiffres, et que mesure-t-elle surtout ? Pourquoi les étapes des chemins latents sont-elles, elles, toujours « proches » ? Le mélange de deux 5 se passe-t-il mieux que celui d'un 1 et d'un 0, et pourquoi ?

In [ ]:
def blend_pixels_21(x0, x1, n_steps):
    """The images (1 - t) x0 + t x1 for t = np.linspace(0, 1, n_steps): an array (n_steps, 784)."""
    raise NotImplementedError("blend_pixels_21")


def blend_codes_21(model, x0, x1, n_steps, method="linear"):
    """Encode x0 and x1, interpolate their codes with mylearn.vae.interpolate_latents(..., n_steps, method=method),
    and decode: an array (n_steps, 784)."""
    raise NotImplementedError("blend_codes_21")


def off_manifold_21(model, images):
    """How far each image is from what the autoencoder can redraw: the MSE between each image and its
    reconstruction, an array of len(images) values."""
    raise NotImplementedError("off_manifold_21")

In [ ]:
with checked("25.21"):
    toy_a_21, toy_b_21 = np.zeros(784, dtype=np.float32), np.ones(784, dtype=np.float32)
    mix_21 = blend_pixels_21(toy_a_21, toy_b_21, 5)
    if returned("25.21", "blend_pixels_21", mix_21):
        same("25.21", "blend_pixels_21(noir, blanc, 5)[:, 0]", np.asarray(mix_21)[:, 0], [0, 0.25, 0.5, 0.75, 1], rtol=1e-6)
    x0_21, x1_21 = XM_TEST[np.flatnonzero(YM_TEST == 1)[0]], XM_TEST[np.flatnonzero(YM_TEST == 0)[0]]
    codes_21 = blend_codes_21(AE_DEEP, x0_21, x1_21, 6)
    if returned("25.21", "blend_codes_21", codes_21):
        z_21 = encode(AE_DEEP, np.stack([x0_21, x1_21]))
        same("25.21", "blend_codes_21(AE_DEEP, un 1, un 0, 6) : les extrémités sont les reconstructions",
             np.asarray(codes_21)[[0, -1]], decode(AE_DEEP, z_21), rtol=1e-4, atol=1e-5)
    off_21 = off_manifold_21(AE_DEEP, XM_TEST[:3])
    if returned("25.21", "off_manifold_21", off_21):
        same("25.21", "off_manifold_21(AE_DEEP, 3 chiffres)", off_21,
             ((reconstruct(AE_DEEP, XM_TEST[:3]) - XM_TEST[:3]) ** 2).mean(axis=1), rtol=1e-5, atol=1e-7)

In [ ]:
with checked("25.21"):
    def _pick_21(digit, k=0):
        return XM_TEST[np.flatnonzero(YM_TEST == digit)[k]]
    for (a_21, b_21) in [(_pick_21(1), _pick_21(0)), (_pick_21(5, 0), _pick_21(5, 3))]:
        rows_21 = {"pixels": blend_pixels_21(a_21, b_21, 7), "codes, ligne droite": blend_codes_21(AE_DEEP, a_21, b_21, 7),
                   "codes, slerp": blend_codes_21(AE_DEEP, a_21, b_21, 7, method="slerp")}
        show_rows(list(rows_21.values()), list(rows_21.keys()), size=1.1)
        for name_21, images_21 in rows_21.items():
            print(f"  {name_21:20s} distance à ce que l'AE sait refaire, étape par étape : "
                  + " ".join(f"{v:.3f}" for v in off_manifold_21(AE_DEEP, images_21)))

### Ex 25.22 — Taille du goulot et profondeur : la courbe d'erreur de reconstruction 🔬 ★★★ ⏱️ 40 min
**Objectif :** tracer l'erreur de reconstruction selon la taille du goulot, pour un autoencodeur dense et un profond.  
**Prérequis :** Ex 25.16 · fiche §25.4 · **Fil rouge :** MNIST · **Parcours :** C

La fiche résume la leçon du §25.4 : il faut **assez de latents** et **assez de capacité**. Tu mesures les deux ensemble : l'erreur de test de l'autoencodeur dense (`make_dense_ae`) et du profond (`make_deep_ae`) pour des goulots de 2, 5, 10, 20 et 50 latents.

**Contrat** : `bottleneck_sweep_22(sizes, makers, epochs, seed=2522, x=None)` : pour chaque nom et chaque fonction du dictionnaire `makers`, et chaque taille de `sizes`, `torch.manual_seed(seed)`, puis `model = maker(size)`, `train_ae(model, x, epochs, seed=seed)` (avec `x = X_MNIST` si `x` vaut `None`), puis la MSE moyenne de reconstruction sur `XM_TEST` ; elle rend le dictionnaire `{(nom, taille): MSE}`.

La vérification compare ta fonction à un entraînement fait à la main sur un cas d'essai. La cellule suivante lance les dix entraînements (une quarantaine de secondes en mode rapide, 4 epochs) et trace les deux courbes.

Dans tes notes : comment l'erreur baisse-t-elle quand le goulot grandit (régulièrement, ou de moins en moins) ? Pour quelles tailles la profondeur aide-t-elle le plus ? Avec 4 epochs, quel modèle est désavantagé, et comment le vérifierais-tu ?

In [ ]:
def bottleneck_sweep_22(sizes, makers, epochs, seed=2522, x=None):
    """For each name, maker of the dict `makers` and each size of `sizes`: torch.manual_seed(seed), model =
    maker(size), train_ae(model, x, epochs, seed=seed) (x = X_MNIST if None), then the mean MSE of reconstruction
    on XM_TEST. Returns the dict {(name, size): that MSE}."""
    raise NotImplementedError("bottleneck_sweep_22")

In [ ]:
with checked("25.22"):
    tiny_22 = bottleneck_sweep_22([3], {"dense": make_dense_ae}, epochs=1, seed=7, x=X_MNIST[:512])
    if returned("25.22", "bottleneck_sweep_22", tiny_22):
        torch.manual_seed(7)
        model_22 = make_dense_ae(3)
        train_ae(model_22, X_MNIST[:512], 1, seed=7)
        want_22 = float(mse_per_image(reconstruct(model_22, XM_TEST), XM_TEST).mean())
        verdict("25.22", isinstance(tiny_22, dict) and set(tiny_22) == {("dense", 3)}
                and abs(tiny_22[("dense", 3)] - want_22) < 1e-5,
                f"bottleneck_sweep_22 sur un cas d'essai : {{('dense', 3): {want_22:.4f}}}.",
                f"attendu {{('dense', 3): {want_22:.4f}}}, reçu {tiny_22!r:.200} (torch.manual_seed(seed) juste avant "
                "maker(size), puis train_ae(..., seed=seed)).")

In [ ]:
with checked("25.22"):
    SIZES_22 = (2, 5, 10, 20, 50)
    EPOCHS_22 = 4 if FAST_MODE else 15
    start_22 = time.perf_counter()
    sweep_22 = bottleneck_sweep_22(SIZES_22, {"dense": make_dense_ae, "profond": make_deep_ae}, EPOCHS_22)
    print(f"{len(sweep_22)} entraînements en {time.perf_counter() - start_22:.0f} s")
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for name_22 in ("dense", "profond"):
        ax.plot(SIZES_22, [sweep_22[(name_22, s)] for s in SIZES_22], "o-", label=name_22)
    ax.set_xscale("log")
    ax.set_xticks(SIZES_22, [str(s) for s in SIZES_22])
    ax.set_xlabel("taille du goulot (latents)")
    ax.set_ylabel("MSE de test")
    ax.legend()
    plt.show()
    for s_22 in SIZES_22:
        print(f"{s_22:3d} latents · dense {sweep_22[('dense', s_22)]:.4f} · profond {sweep_22[('profond', s_22)]:.4f}")

### Ex 25.23 — L'autoencodeur linéaire face à l'ACP de scikit-learn 🔬 ★★★ ⏱️ 40 min
**Objectif :** vérifier par l'expérience qu'un autoencodeur linéaire retrouve le sous-espace de l'ACP, mais pas ses axes.  
**Prérequis :** ∂ 25.8 · ch. 12 (ACP) · fiche §25.6 · **Fil rouge :** MNIST · **Parcours :** M, C

Le ∂ 25.8 l'a démontré dans le plan : un autoencodeur linéaire entraîné à l'erreur quadratique retrouve le **sous-espace** de l'ACP, pas ses axes. Tu le vérifies sur MNIST, avec 16 latents. Deux sous-espaces se comparent par leurs **angles principaux** : tous nuls quand les sous-espaces sont les mêmes, quelle que soit la base qui les décrit (`scipy.linalg.subspace_angles`).

**Contrat**
- `fit_linear_ae_23(xc, k, epochs=40, lr=3e-3, batch_size=256, seed=2523)` : un autoencodeur **linéaire** sur les images centrées `xc` `(N, 784)` : encodeur `nn.Linear(784, k, bias=False)` et décodeur `nn.Linear(k, 784, bias=False)`, sans activation, créés dans cet ordre juste après `torch.manual_seed(seed)`, entraînés par `train_ae` avec l'erreur quadratique (`loss="mse"`) et les mêmes `epochs`, `lr`, `batch_size` et `seed` ; elle rend le poids du décodeur en tableau NumPy `(784, k)` : ses $k$ colonnes engendrent le sous-espace appris ;
- `subspace_angles_deg_23(A, B)` rend les angles principaux, en degrés, entre le sous-espace des colonnes de `A` et celui de `B`, du plus grand au plus petit.

La vérification essaie tes angles sur deux plans de l'espace, puis ta fonction d'entraînement sur un petit cas. La cellule suivante centre 10 000 images, calcule l'ACP à 16 composantes (`svd_solver="full"`), entraîne ton autoencodeur (quelques secondes), et compare : les erreurs de reconstruction, les angles entre les sous-espaces, les colonnes du décodeur face aux axes de l'ACP.

Dans tes notes : les deux sous-espaces sont-ils les mêmes ? Pourquoi les plus grands angles sont-ils ceux des dernières directions (pense aux valeurs propres voisines) ? Les colonnes du décodeur sont-elles les axes de l'ACP, et sont-elles orthogonales ? Pourquoi un test « mêmes axes » aurait-il échoué ?

In [ ]:
def fit_linear_ae_23(xc, k, epochs=40, lr=3e-3, batch_size=256, seed=2523):
    """A LINEAR autoencoder on the centred images xc (N, 784): encoder nn.Linear(784, k, bias=False), decoder
    nn.Linear(k, 784, bias=False), no activation, both created just after torch.manual_seed(seed); trained with
    Adam(lr) on the mean squared error, mini-batches in an order drawn by ONE torch.Generator().manual_seed(seed).
    Returns the decoder's weight as a NumPy array (784, k): its k columns span the learned subspace."""
    raise NotImplementedError("fit_linear_ae_23")


def subspace_angles_deg_23(A, B):
    """The principal angles, in degrees, between the subspace spanned by the columns of A and that of B (both
    (p, k)), from the largest to the smallest: scipy.linalg.subspace_angles, converted."""
    raise NotImplementedError("subspace_angles_deg_23")

In [ ]:
with checked("25.23"):
    plane_a_23 = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])                       # the plane z = 0
    tilt_23 = np.radians(30)
    plane_b_23 = np.array([[1.0, 0.0], [0.0, np.cos(tilt_23)], [0.0, np.sin(tilt_23)]])  # the same, tilted by 30°
    ang_23 = subspace_angles_deg_23(plane_a_23, plane_b_23)
    if returned("25.23", "subspace_angles_deg_23", ang_23):
        same("25.23", "subspace_angles_deg_23(plan z = 0, le même incliné de 30°)", ang_23, [30.0, 0.0], atol=1e-9)
        same("25.23", "deux bases différentes du même plan", subspace_angles_deg_23(plane_a_23, plane_a_23 @ [[2.0, 1.0], [1.0, 3.0]]),
             [0.0, 0.0], atol=1e-6)
    xc_23 = X_MNIST[:2000] - X_MNIST[:2000].mean(axis=0)
    W_23 = fit_linear_ae_23(xc_23, 4, epochs=2)
    if returned("25.23", "fit_linear_ae_23", W_23):
        torch.manual_seed(2523)
        enc_23, dec_23 = nn.Linear(784, 4, bias=False), nn.Linear(4, 784, bias=False)
        train_ae(AutoEncoder(enc_23, dec_23), xc_23, 2, lr=3e-3, batch_size=256, seed=2523, loss="mse")
        same("25.23", "fit_linear_ae_23(2 000 images centrées, k=4, epochs=2) : la matrice du décodeur (784, 4)",
             W_23, dec_23.weight.detach().cpu().numpy(), rtol=1e-4, atol=1e-6)

In [ ]:
with checked("25.23"):
    K_23 = 16
    x_23 = X_MNIST[:N_FIXED].astype(np.float64)
    xc_23 = x_23 - x_23.mean(axis=0)                          # centred with the mean of the training images
    pca_23 = PCA(n_components=K_23, svd_solver="full").fit(xc_23)
    P_23 = pca_23.components_.T                               # (784, K): the first K principal axes
    W_23 = fit_linear_ae_23(xc_23.astype(np.float32), K_23)
    def _rec_mse_23(basis):
        Q, _ = np.linalg.qr(basis)                            # an orthonormal basis of the same subspace
        return float(((xc_23 - xc_23 @ Q @ Q.T) ** 2).mean())
    angles_23 = subspace_angles_deg_23(W_23, P_23)
    print(f"MSE de reconstruction · ACP {_rec_mse_23(P_23):.5f} · autoencodeur linéaire {_rec_mse_23(W_23):.5f}")
    print(f"angles entre les deux sous-espaces (degrés) : max {angles_23.max():.1f} · moyenne {angles_23.mean():.1f}")
    unit_23 = W_23 / np.linalg.norm(W_23, axis=0)
    print("|cos| entre chaque colonne du décodeur et l'axe de l'ACP de même rang :",
          np.round(np.abs((unit_23 * P_23).sum(axis=0)), 2))
    gram_23 = unit_23.T @ unit_23
    print(f"colonnes du décodeur orthogonales ? plus grand |cos| entre deux d'entre elles : "
          f"{np.abs(gram_23 - np.eye(K_23)).max():.2f}")
    show_rows([P_23.T[:8] / np.abs(P_23.T[:8]).max(axis=1, keepdims=True) / 2 + 0.5,
               unit_23.T[:8] / np.abs(unit_23.T[:8]).max(axis=1, keepdims=True) / 2 + 0.5],
              ["axes de l'ACP", "colonnes du décodeur"], "les 8 premières directions (gris = 0)", size=1.1)

## Partie B · Autoencodeurs convolutifs et débruiteurs

Tu construis l'autoencodeur convolutif du livre (25.24), tu répares un débruiteur qui apprenait le bruit (25.25), tu débruites Fashion-MNIST (25.26), puis tu remplaces pooling et sur-échantillonnage par des convolutions à pas (25.27). La cellule suivante fixe le bruit des débruiteurs de vêtements ($\sigma = 0{,}3$), leurs images d'entraînement (`N_DENOISE` : 10 000 en mode rapide) et 1 000 vêtements de validation avec leur version bruitée (un bruit fixé), et donne `layout`, qui écrit brièvement les couches d'un `nn.Sequential` (les vérifications comparent ces listes).

In [ ]:
SIGMA_B = 0.3                                   # the noise of the Fashion-MNIST denoisers (25.26, 25.27, 25.39)
N_DENOISE = 10_000 if FAST_MODE else 60_000      # their training images
VAL_CLEAN_B = XF_TEST[:1000]                     # 1 000 validation clothes, and their noisy version (a fixed noise)
VAL_NOISY_B = np.clip(VAL_CLEAN_B + SIGMA_B * np.random.default_rng(2526).standard_normal(VAL_CLEAN_B.shape),
                      0, 1).astype(np.float32)


def layout(seq):
    """The layers of a nn.Sequential, written briefly: what the checks compare."""
    out = []
    for m in seq:
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            extra = f", op={m.output_padding[0]}" if isinstance(m, nn.ConvTranspose2d) else ""
            out.append(f"{type(m).__name__}({m.in_channels}, {m.out_channels}, k={m.kernel_size[0]}, "
                       f"s={m.stride[0]}, p={m.padding[0]}{extra})")
        elif isinstance(m, nn.Linear):
            out.append(f"Linear({m.in_features}, {m.out_features})")
        elif isinstance(m, nn.Upsample):
            out.append(f"Upsample({m.scale_factor:g})")
        elif isinstance(m, nn.MaxPool2d):
            out.append(f"MaxPool2d({m.kernel_size})")
        elif isinstance(m, nn.Unflatten):
            out.append(f"Unflatten{tuple(m.unflattened_size)}")
        else:
            out.append(type(m).__name__)
    return out

### Ex 25.24 — Autoencodeur convolutif à 147 latents 📦 ★★ ⏱️ 30 min
**Objectif :** écrire l'autoencodeur convolutif du livre et comparer son code de 147 valeurs à celui de l'autoencodeur dense.  
**Prérequis :** ✏️ 25.3 · ch. 21 (convolutions, pooling) · fiche §25.7, §25.7.1, §25.7.2 · **Fil rouge :** MNIST · **Parcours :** R, C

Le ✏️ 25.3 a suivi les formes de l'autoencodeur convolutif du livre ; tu l'écris. Pour garder la convention du notebook (des lignes de 784 pixels), l'encodeur commence par `nn.Unflatten(1, (1, 28, 28))` et le décodeur finit par `nn.Flatten()`.

Une différence avec le livre : il met une ReLU après la dernière convolution de l'encodeur, sur le code ; ton code n'a pas d'activation, sauf si tu demandes la version du livre.

**Contrat** : `make_conv_ae_24(relu_code=False)` rend un `AutoEncoder` dont l'encodeur est `nn.Unflatten(1, (1, 28, 28))`, `Conv2d(1, 16, 3, padding=1)`, ReLU, `MaxPool2d(2)`, `Conv2d(16, 8, 3, padding=1)`, ReLU, `MaxPool2d(2)`, `Conv2d(8, 3, 3, padding=1)`, suivi d'une ReLU seulement si `relu_code` vaut `True`, et le décodeur `nn.Upsample(scale_factor=2)`, `Conv2d(3, 16, 3, padding=1)`, ReLU, `nn.Upsample(scale_factor=2)`, `Conv2d(16, 1, 3, padding=1)`, sigmoïde, `nn.Flatten()`.

La vérification compare tes couches au contrat et regarde les formes du code et de la sortie. La cellule suivante entraîne les deux versions, sans et avec la ReLU du livre (3 epochs chacune en mode rapide, une trentaine de secondes en tout), compare leurs erreurs à celle de `AE_DEEP`, et décode une ligne droite entre les codes d'un 1 et d'un 0.

Dans tes notes : combien de paramètres a ce modèle, face au 1,1 million de `AE_DEEP`, et son code est-il plus petit ou plus grand ? Que s'est-il passé avec la ReLU sur le code (regarde la part des valeurs nulles, et pense au gradient d'une ReLU en dessous de 0) ? Que donne le mélange des codes : une transformation, ou un fondu (fiche §25.7.1) ?

In [ ]:
def make_conv_ae_24(relu_code=False):
    """The book's convolutional autoencoder, for rows of 784 pixels:
    encoder: nn.Unflatten(1, (1, 28, 28)), Conv2d(1, 16, 3, padding=1), ReLU, MaxPool2d(2),
             Conv2d(16, 8, 3, padding=1), ReLU, MaxPool2d(2), Conv2d(8, 3, 3, padding=1)        -> a code (N, 3, 7, 7)
             (then a last ReLU on the code only if relu_code is True, as in the book)
    decoder: nn.Upsample(scale_factor=2), Conv2d(3, 16, 3, padding=1), ReLU, nn.Upsample(scale_factor=2),
             Conv2d(16, 1, 3, padding=1), Sigmoid, nn.Flatten()                              -> rows of 784 pixels
    Returns an AutoEncoder whose encoder and decoder are these two nn.Sequential."""
    raise NotImplementedError("make_conv_ae_24")

In [ ]:
with checked("25.24"):
    torch.manual_seed(0)
    model_24 = make_conv_ae_24()
    if returned("25.24", "make_conv_ae_24", model_24):
        want_enc_24 = ["Unflatten(1, 28, 28)", "Conv2d(1, 16, k=3, s=1, p=1)", "ReLU", "MaxPool2d(2)",
                       "Conv2d(16, 8, k=3, s=1, p=1)", "ReLU", "MaxPool2d(2)", "Conv2d(8, 3, k=3, s=1, p=1)"]
        want_dec_24 = ["Upsample(2)", "Conv2d(3, 16, k=3, s=1, p=1)", "ReLU", "Upsample(2)",
                       "Conv2d(16, 1, k=3, s=1, p=1)", "Sigmoid", "Flatten"]
        verdict("25.24", layout(model_24.encoder) == want_enc_24, "l'encodeur suit le contrat.",
                f"encodeur attendu {want_enc_24}, reçu {layout(model_24.encoder)}.")
        verdict("25.24", layout(make_conv_ae_24(relu_code=True).encoder) == want_enc_24 + ["ReLU"],
                "make_conv_ae_24(relu_code=True) ajoute la ReLU du livre sur le code.",
                "avec relu_code=True, l'encodeur doit finir par une ReLU de plus (la version du livre).")
        verdict("25.24", layout(model_24.decoder) == want_dec_24, "le décodeur suit le contrat.",
                f"décodeur attendu {want_dec_24}, reçu {layout(model_24.decoder)}.")
        code_24 = encode(model_24, X_MNIST[:5])
        verdict("25.24", code_24.shape == (5, 3, 7, 7) and reconstruct(model_24, X_MNIST[:5]).shape == (5, 784),
                f"un code {code_24.shape} pour 5 images, une sortie (5, 784).",
                f"attendu un code (5, 3, 7, 7) et une sortie (5, 784), reçu un code {code_24.shape}.")

In [ ]:
with checked("25.24"):
    EPOCHS_24 = 3 if FAST_MODE else 10
    convs_24 = {}
    for relu_24 in (False, True):
        torch.manual_seed(2524)
        model_24 = make_conv_ae_24(relu_code=relu_24)
        history_24 = train_ae(model_24, X_MNIST, EPOCHS_24, seed=2524, x_val=XM_TEST)
        convs_24[relu_24] = model_24
        print(f"convolutif{' avec la ReLU du livre sur le code' if relu_24 else ', code sans activation':40s} "
              f"{n_params(model_24):,d} paramètres · MSE de test par epoch : "
              + " ".join(f"{v:.4f}" for v in history_24["val_mse"]) + f" ({sum(history_24['seconds']):.0f} s)")
    print(f"AE_DEEP (20 latents, 8 epochs) : MSE de test {mse_per_image(reconstruct(AE_DEEP, XM_TEST), XM_TEST).mean():.4f}")
    codes_relu_24 = encode(convs_24[True], XM_TEST[:500])
    print(f"avec la ReLU : part des valeurs du code exactement nulles {np.mean(codes_relu_24 == 0):.0%}")
    conv_24 = convs_24[False]
    x0_24, x1_24 = XM_TEST[np.flatnonzero(YM_TEST == 1)[0]], XM_TEST[np.flatnonzero(YM_TEST == 0)[0]]
    c0_24, c1_24 = encode(conv_24, np.stack([x0_24, x1_24])).reshape(2, -1)        # codes flattened to 147 values
    path_24 = np.stack([(1 - t) * c0_24 + t * c1_24 for t in np.linspace(0, 1, 7)]).reshape(7, 3, 7, 7)
    show_rows([XM_TEST[:8], reconstruct(conv_24, XM_TEST[:8]), reconstruct(convs_24[True], XM_TEST[:8]),
               decode(conv_24, path_24.astype(np.float32))],
              ["originaux", "code sans activation", "ReLU sur le code", "d'un 1 à un 0"],
              f"{EPOCHS_24} epochs sur {len(X_MNIST)} images")

### Ex 25.25 — Le débruiteur qui apprenait le bruit 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver les trois erreurs d'une boucle d'entraînement de débruiteur, et la corriger.  
**Prérequis :** Ex 25.24 · fiche §25.8 · **Fil rouge :** MNIST · **Parcours :** C

Un collègue a écrit la boucle d'entraînement d'un débruiteur (la cellule suivante), et ses images « débruitées » restent pleines de neige. Trois lignes sont fausses. Les diagnostics de la cellule donnée l'essaient avec un faux modèle qui note ce qu'il reçoit (pas d'entraînement réel : ils sont instantanés), dans cet ordre : 1) la cible est-elle l'image propre ? 2) une même image reçoit-elle un bruit neuf à chaque epoch ? 3) les entrées bruitées restent-elles dans $[0, 1]$ ?

1. Écris dans `faulty_25` les lettres des **lignes fautives** (une chaîne comme `"AB"`).
2. Écris `train_denoiser_fixed_25`, la version corrigée : mêmes arguments ; un seul `torch.Generator` créé depuis `seed`, un `torch.randperm(N, generator=g)` au début de chaque epoch ; pour chaque mini-batch, un bruit **neuf** `sigma * torch.randn(batch.shape, generator=g)`, des entrées bruitées **écrêtées** dans $[0, 1]$ (`torch.clamp`), et la loss `loss_fn(model(entrées bruitées), batch propre)` ; la fonction rend le modèle.

La vérification lance les trois diagnostics sur ta version ; s'ils passent, elle entraîne un petit débruiteur dense avec ta fonction et montre le résultat.

Dans tes notes : pour chaque erreur, qu'apprendrait le réseau, et comment le verrais-tu sur les images ou sur les courbes ? Laquelle est la plus grave ?

In [ ]:
# The code to debug (do not change this cell: copy the function into the next one to fix it)
def train_denoiser_25(model, x, epochs, sigma=0.5, lr=1e-3, batch_size=128, seed=0, loss_fn=F.binary_cross_entropy):
    """Train `model` to remove a gaussian noise of standard deviation sigma from the images x (N, 784) in [0, 1]."""
    x = to_tensor(x)                                                   # (A)
    g = torch.Generator().manual_seed(seed)                            # (B)
    noise = sigma * torch.randn(x.shape, generator=g)                  # (C)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)            # (D)
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=g)                    # (E)
        for i in range(0, len(x), batch_size):
            idx = order[i:i + batch_size]
            inputs = x[idx] + noise[idx]                               # (F)
            loss = loss_fn(model(inputs.to(DEVICE)), inputs.to(DEVICE))   # (G)
            optimizer.zero_grad()
            loss.backward()                                            # (H)
            optimizer.step()
    return model


class SpyModel_25(nn.Module):
    """A fake model for the checks: it remembers the inputs it receives, and returns a constant grey image."""

    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(1))
        self.inputs = []

    def forward(self, x):
        self.inputs.append(x.detach().cpu().clone())
        return torch.sigmoid(self.w).expand_as(x)


def spy_run_25(train_fn, epochs=2, sigma=1.0):
    """Run train_fn on 8 distinct images with a SpyModel_25 and a loss that remembers its targets:
    returns (the images, the inputs seen, the targets seen), one tensor per mini-batch."""
    x = np.random.default_rng(25).uniform(0.2, 0.8, size=(8, 784)).astype(np.float32)
    targets = []

    def spy_loss(output, target):
        targets.append(target.detach().cpu().clone())
        return F.binary_cross_entropy(output, target.clamp(0, 1))

    spy = SpyModel_25()
    train_fn(spy, x, epochs, sigma=sigma, batch_size=4, seed=3, loss_fn=spy_loss)
    return torch.from_numpy(x), spy.inputs, targets


def diagnose_25(train_fn):
    """The three diagnostics of 25.25, in order: the target, the noise of each epoch, the clipping.
    Prints ✅ or ❌ for each one, stops at the first ❌ (⏳ for the next ones)."""
    x, inputs, targets = spy_run_25(train_fn)
    points = [
        ("la cible est l'image propre",
         all(any(torch.equal(row, clean) for clean in x) for batch in targets for row in batch),
         "la cible d'un mini-batch doit être l'image propre, pas l'entrée bruitée."),
        ("un bruit neuf à chaque epoch",
         not torch.equal(torch.cat(inputs[:len(inputs) // 2]).sort(dim=0).values,
                         torch.cat(inputs[len(inputs) // 2:]).sort(dim=0).values),
         "une même image reçoit le même bruit à chaque epoch : le réseau peut apprendre le bruit par cœur."),
        ("des entrées dans [0, 1]",
         all(float(b.min()) >= 0 and float(b.max()) <= 1 for b in inputs),
         "les entrées bruitées sortent de [0, 1] : le réseau voit des pixels qui n'existent pas (écrête-les)."),
    ]
    for k, (name, ok, why) in enumerate(points, start=1):
        if ok:
            print(f"✅ Ex 25.25 : point {k}, {name}.")
        else:
            print(f"❌ Ex 25.25 : point {k} : {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex 25.25 : point {later} : corrige d'abord le point {k}.")
            return False
    return True


print("Le code fautif, essayé par les diagnostics :")
_ = diagnose_25(train_denoiser_25)

In [ ]:
faulty_25 = ...   # the letters of the faulty lines, e.g. "AB"


def train_denoiser_fixed_25(model, x, epochs, sigma=0.5, lr=1e-3, batch_size=128, seed=0,
                            loss_fn=F.binary_cross_entropy):
    """The fixed version of train_denoiser_25: same arguments, same mini-batches (one torch.Generator from `seed`,
    a torch.randperm(N, generator=g) at the start of each epoch). For each mini-batch: a new noise
    sigma * torch.randn(batch.shape, generator=g), noisy inputs clipped to [0, 1] (torch.clamp), and the loss
    loss_fn(model(noisy inputs), clean batch). Returns the model."""
    raise NotImplementedError("train_denoiser_fixed_25")

In [ ]:
wb.check("25.25", faulty_25)
with checked("25.25"):
    if written(lambda: train_denoiser_fixed_25(SpyModel_25(), np.zeros((1, 784), np.float32), 0)):
        if diagnose_25(train_denoiser_fixed_25):
            torch.manual_seed(2525)
            small_25 = make_deep_ae(32, hidden=(256,))
            train_denoiser_fixed_25(small_25.to(DEVICE), X_MNIST[:5000], 3 if FAST_MODE else 10, sigma=0.5, seed=2525)
            noisy_25 = np.clip(XM_TEST[:8] + 0.5 * np.random.default_rng(25).standard_normal((8, 784)), 0, 1).astype(np.float32)
            show_rows([XM_TEST[:8], noisy_25, reconstruct(small_25, noisy_25)], ["propres", "bruitées (σ = 0,5)", "débruitées"],
                      "un petit débruiteur dense entraîné par ta fonction")
    else:
        print("⏳ Ex 25.25 : les diagnostics attendent ta fonction corrigée.")

La cellule suivante donne la version de référence de la boucle corrigée, `train_denoiser`, et `psnr`, qui servent aux exercices suivants.

In [ ]:
# The reference version of your fixed loop (25.25), with a history: the denoisers of 25.26, 25.27 and 25.39 use it
def train_denoiser(model, x, epochs, sigma=SIGMA_B, lr=1e-3, batch_size=128, seed=0, x_val=None, verbose=False):
    """Train the autoencoder `model` in place (on DEVICE) to turn noisy images back into the clean images x (N, 784):
    a new gaussian noise of standard deviation sigma for each mini-batch, noisy inputs clipped to [0, 1], the binary
    cross-entropy against the CLEAN batch, Adam(lr). x_val = (noisy, clean) adds "val_psnr" to the history."""
    model.to(DEVICE)
    x = to_tensor(x)
    g = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_psnr": [], "seconds": []}
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        total = 0.0
        order = torch.randperm(len(x), generator=g)
        for i in range(0, len(x), batch_size):
            clean = x[order[i:i + batch_size]]
            inputs = torch.clamp(clean + sigma * torch.randn(clean.shape, generator=g), 0, 1)
            loss = F.binary_cross_entropy(model(inputs.to(DEVICE)), clean.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(clean)
        history["loss"].append(total / len(x))
        history["seconds"].append(time.perf_counter() - start)
        if x_val is not None:
            history["val_psnr"].append(psnr(x_val[1], reconstruct(model, x_val[0])))
        if verbose:
            print(f"epoch {epoch:2d} · loss {history['loss'][-1]:.4f}" +
                  (f" · val PSNR {history['val_psnr'][-1]:.2f} dB" if x_val is not None else ""))
    return history


def psnr(clean, images):
    """The mean PSNR (dB) of images against clean (pixels in [0, 1]): the mean over the images of 10 log10(1 / MSE)."""
    return float(np.mean(10 * np.log10(1.0 / np.maximum(mse_per_image(images, clean), 1e-12))))

### Ex 25.26 — Autoencodeur débruiteur (bruit gaussien, écrêtage) 📦 ★★ ⏱️ 30 min
**Objectif :** construire un débruiteur convolutif pour Fashion-MNIST et mesurer son gain en PSNR.  
**Prérequis :** Ex 25.24 · Ex 25.25 · fiche §25.8 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le débruiteur du livre (celui de F. Chollet) garde 32 filtres à chaque étage : deux convolutions suivies d'un max pooling à l'encodage, et leur miroir au décodage. Tu l'entraînes sur des vêtements avec un bruit gaussien d'écart-type 0,3, et tu le mesures en **PSNR**.

**Contrat**
- `make_denoiser_26(channels=32)` rend un `AutoEncoder` ; avec $c$ = `channels`, l'encodeur est `nn.Unflatten(1, (1, 28, 28))`, `Conv2d(1, c, 3, padding=1)`, ReLU, `MaxPool2d(2)`, `Conv2d(c, c, 3, padding=1)`, ReLU, `MaxPool2d(2)`, et le décodeur `Conv2d(c, c, 3, padding=1)`, ReLU, `nn.Upsample(scale_factor=2)`, `Conv2d(c, c, 3, padding=1)`, ReLU, `nn.Upsample(scale_factor=2)`, `Conv2d(c, 1, 3, padding=1)`, sigmoïde, `nn.Flatten()` ;
- `psnr_26(clean, images)` rend le PSNR moyen en décibels de `images` face à `clean` (deux tableaux `(N, 784)`, pixels dans $[0, 1]$) : pour chaque image $10 \log_{10}(1/\text{MSE})$, puis la moyenne sur les images.

La vérification essaie ton architecture avec 8 filtres et ton PSNR sur deux images d'erreurs connues. La cellule suivante entraîne ton débruiteur avec `train_denoiser` (3 epochs sur 10 000 vêtements en mode rapide, une trentaine de secondes, avec un learning rate de 0,003), compare le PSNR des images bruitées et débruitées, et les montre.

Dans tes notes : combien de décibels gagne le débruiteur, et à quoi cela correspond-il en MSE (fiche : diviser la MSE par 10 ajoute 10 dB) ? Quels détails perd-il ? Pourquoi moyenne-t-on le PSNR image par image plutôt que de calculer celui de la MSE moyenne ?

In [ ]:
def make_denoiser_26(channels=32):
    """A convolutional denoiser for rows of 784 pixels, with c = channels:
    encoder: nn.Unflatten(1, (1, 28, 28)), Conv2d(1, c, 3, padding=1), ReLU, MaxPool2d(2),
             Conv2d(c, c, 3, padding=1), ReLU, MaxPool2d(2)                         -> (N, c, 7, 7)
    decoder: Conv2d(c, c, 3, padding=1), ReLU, nn.Upsample(scale_factor=2), Conv2d(c, c, 3, padding=1), ReLU,
             nn.Upsample(scale_factor=2), Conv2d(c, 1, 3, padding=1), Sigmoid, nn.Flatten()
    Returns an AutoEncoder."""
    raise NotImplementedError("make_denoiser_26")


def psnr_26(clean, images):
    """The mean PSNR in decibels of `images` against `clean` (both (N, 784), pixels in [0, 1]): for each image,
    10 log10(1 / MSE), then the mean over the images. A float."""
    raise NotImplementedError("psnr_26")

In [ ]:
with checked("25.26"):
    torch.manual_seed(0)
    model_26 = make_denoiser_26(8)
    if returned("25.26", "make_denoiser_26", model_26):
        want_26 = (["Unflatten(1, 28, 28)", "Conv2d(1, 8, k=3, s=1, p=1)", "ReLU", "MaxPool2d(2)",
                    "Conv2d(8, 8, k=3, s=1, p=1)", "ReLU", "MaxPool2d(2)"],
                   ["Conv2d(8, 8, k=3, s=1, p=1)", "ReLU", "Upsample(2)", "Conv2d(8, 8, k=3, s=1, p=1)", "ReLU",
                    "Upsample(2)", "Conv2d(8, 1, k=3, s=1, p=1)", "Sigmoid", "Flatten"])
        verdict("25.26", (layout(model_26.encoder), layout(model_26.decoder)) == want_26,
                "make_denoiser_26(8) suit le contrat.",
                f"attendu {want_26}, reçu {(layout(model_26.encoder), layout(model_26.decoder))}.")
    toy_clean_26 = np.zeros((2, 784))
    toy_26 = toy_clean_26 + np.array([[0.1], [0.01]])          # MSE 0.01 then 0.0001: 20 dB then 40 dB
    value_26 = psnr_26(toy_clean_26, toy_26)
    if returned("25.26", "psnr_26", value_26):
        same("25.26", "psnr_26 de deux images d'erreurs 0,1 et 0,01 sur chaque pixel (20 et 40 dB)", [value_26], [30.0],
             rtol=1e-9)

In [ ]:
with checked("25.26"):
    EPOCHS_26 = 3 if FAST_MODE else 10
    torch.manual_seed(2526)
    denoiser_26 = make_denoiser_26(32)
    history_26 = train_denoiser(denoiser_26, X_FASHION[:N_DENOISE], EPOCHS_26, lr=3e-3, seed=2526)
    out_26 = reconstruct(denoiser_26, VAL_NOISY_B)
    print(f"{n_params(denoiser_26):,d} paramètres · {sum(history_26['seconds']):.0f} s · PSNR des images bruitées "
          f"{psnr_26(VAL_CLEAN_B, VAL_NOISY_B):.2f} dB · débruitées {psnr_26(VAL_CLEAN_B, out_26):.2f} dB")
    verdict("25.26", psnr_26(VAL_CLEAN_B, out_26) > psnr_26(VAL_CLEAN_B, VAL_NOISY_B) + 5,
            "le débruiteur gagne plus de 5 dB.", "le débruiteur devrait gagner plus de 5 dB : vérifie ton architecture.")
    show_rows([VAL_CLEAN_B[:8], VAL_NOISY_B[:8], out_26[:8]], ["propres", f"bruitées (σ = {SIGMA_B})", "débruitées"])

### Ex 25.27 — Sur-échantillonnage explicite ou convolutions à pas : temps et qualité 🔬 ★★★ ⏱️ 35 min
**Objectif :** comparer, à nombre de filtres égal, un débruiteur à pooling et sur-échantillonnage et un débruiteur à convolutions à pas.  
**Prérequis :** Ex 25.26 · ✏️ 25.4 · fiche §25.8 (et son encadré ⚠️) · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Le livre remplace le pooling et le sur-échantillonnage de son débruiteur par des convolutions à pas : un `Conv2d` de stride 2 réduit l'image, un `ConvTranspose2d` de stride 2 l'agrandit (le ✏️ 25.4 a calculé ses tailles). Tu mesures ce que cela change, au même nombre de filtres.

**Contrat**
- `make_strided_27(channels=32)` rend un `AutoEncoder` ; l'encodeur est `nn.Unflatten(1, (1, 28, 28))`, `Conv2d(1, c, 3, stride=2, padding=1)`, ReLU, `Conv2d(c, c, 3, stride=2, padding=1)`, ReLU ; le décodeur `ConvTranspose2d(c, c, 3, stride=2, padding=1, output_padding=1)`, ReLU, le même encore, ReLU, puis `Conv2d(c, 1, 3, padding=1)`, sigmoïde, `nn.Flatten()` ;
- `compare_27(makers, epochs, seed=2527)` : pour chaque nom et chaque fonction du dictionnaire `makers`, `torch.manual_seed(seed)`, `model = maker()`, `train_denoiser(model, X_FASHION[:N_DENOISE], epochs, seed=seed)` ; elle rend `{nom: {"params": ..., "sec_per_epoch": la médiane des durées, "psnr": le PSNR moyen sur VAL_NOISY_B face à VAL_CLEAN_B}}`.

La vérification compare tes couches au contrat, puis `compare_27` à un entraînement fait à la main. La cellule suivante entraîne les deux débruiteurs (2 epochs en mode rapide, une trentaine de secondes ; 🚀 le mode complet est long sur un CPU).

Dans tes notes : lequel va le plus vite, et pourquoi (compte les positions où chaque convolution calcule) ? Lequel débruite le mieux à nombre d'epochs égal ? Une convolution transposée peut laisser un motif en damier : le vois-tu ici ?

In [ ]:
def make_strided_27(channels=32):
    """The same denoiser without pooling nor upsampling, with c = channels:
    encoder: nn.Unflatten(1, (1, 28, 28)), Conv2d(1, c, 3, stride=2, padding=1), ReLU,
             Conv2d(c, c, 3, stride=2, padding=1), ReLU                                  -> (N, c, 7, 7)
    decoder: ConvTranspose2d(c, c, 3, stride=2, padding=1, output_padding=1), ReLU,
             ConvTranspose2d(c, c, 3, stride=2, padding=1, output_padding=1), ReLU,
             Conv2d(c, 1, 3, padding=1), Sigmoid, nn.Flatten()
    Returns an AutoEncoder."""
    raise NotImplementedError("make_strided_27")


def compare_27(makers, epochs, seed=2527):
    """For each name, maker of the dict `makers`: torch.manual_seed(seed), model = maker(), train_denoiser(model,
    X_FASHION[:N_DENOISE], epochs, seed=seed). Returns {name: {"params": n_params(model), "sec_per_epoch": the
    median of history["seconds"], "psnr": the mean PSNR of model on VAL_NOISY_B against VAL_CLEAN_B}}."""
    raise NotImplementedError("compare_27")

In [ ]:
with checked("25.27"):
    torch.manual_seed(0)
    model_27 = make_strided_27(8)
    if returned("25.27", "make_strided_27", model_27):
        want_27 = (["Unflatten(1, 28, 28)", "Conv2d(1, 8, k=3, s=2, p=1)", "ReLU", "Conv2d(8, 8, k=3, s=2, p=1)", "ReLU"],
                   ["ConvTranspose2d(8, 8, k=3, s=2, p=1, op=1)", "ReLU", "ConvTranspose2d(8, 8, k=3, s=2, p=1, op=1)",
                    "ReLU", "Conv2d(8, 1, k=3, s=1, p=1)", "Sigmoid", "Flatten"])
        verdict("25.27", (layout(model_27.encoder), layout(model_27.decoder)) == want_27,
                "make_strided_27(8) suit le contrat.",
                f"attendu {want_27}, reçu {(layout(model_27.encoder), layout(model_27.decoder))}.")
        verdict("25.27", encode(model_27, X_FASHION[:2]).shape == (2, 8, 7, 7) and reconstruct(model_27, X_FASHION[:2]).shape == (2, 784),
                "un code (2, 8, 7, 7) et une sortie (2, 784).", "le code doit être (2, 8, 7, 7) et la sortie (2, 784).")
    tiny_27 = compare_27({"petit": lambda: make_strided_27(4)}, 1, seed=3)
    if returned("25.27", "compare_27", tiny_27):
        torch.manual_seed(3)
        ref_27 = make_strided_27(4)
        train_denoiser(ref_27, X_FASHION[:N_DENOISE], 1, seed=3)
        row_27 = tiny_27.get("petit", {}) if isinstance(tiny_27, dict) else {}
        verdict("25.27", set(row_27) == {"params", "sec_per_epoch", "psnr"} and row_27["params"] == n_params(ref_27)
                and abs(row_27["psnr"] - psnr(VAL_CLEAN_B, reconstruct(ref_27, VAL_NOISY_B))) < 1e-3,
                "compare_27 sur un cas d'essai : les paramètres et le PSNR attendus.",
                f"compare_27 doit rendre {{nom: {{'params', 'sec_per_epoch', 'psnr'}}}} avec les valeurs d'un entraînement "
                f"fait comme le contrat, reçu {tiny_27!r:.200}.")

In [ ]:
with checked("25.27"):
    EPOCHS_27 = 2 if FAST_MODE else 10                 # 🚀 the full mode trains for long: a GPU helps
    table_27 = compare_27({"pooling + sur-échantillonnage": lambda: make_denoiser_26(32),
                           "convolutions à pas": lambda: make_strided_27(32)}, EPOCHS_27)
    for name_27, row_27 in table_27.items():
        print(f"{name_27:30s} {row_27['params']:7,d} paramètres · {row_27['sec_per_epoch']:5.2f} s par epoch · "
              f"PSNR {row_27['psnr']:.2f} dB")

## Partie C · Le VAE

Tu écris dans ta librairie la reparamétrisation et la KL gaussienne (25.28, 25.29), puis un VAE en PyTorch (25.30). Tu prédis ce que deviennent des codes tirés au hasard (25.31), tu explores la stochasticité du VAE (25.32) et son espace latent en 2D (25.33), tu lis des courbes d'entraînement (25.34), tu exportes le décodeur comme générateur (25.35), tu règles $\beta$ (25.36), tu détectes des anomalies (25.37), tu écris un VAE entièrement en NumPy (25.38), et tu relèves le défi de débruitage (25.39).

### Ex 25.28 — La reparamétrisation en NumPy : passe avant et passe arrière 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire le tirage $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}$ et sa passe arrière dans ta librairie.  
**Prérequis :** ∂ 25.5 · ch. 18 (passe arrière) · fiche §25.9.2 · **mylearn :** `vae.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/vae.py` (créé par `python tools/start_chapter.py 25`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur quelques cas ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_ch25_vae.py -q` les lance tous dans un terminal.

**Contrat commun de `vae.py`** (les docstrings, plus ces précisions que vérifient les tests) : les entrées peuvent être des listes et ne sont jamais modifiées ; chaque tableau rendu est un **nouveau** tableau `float64`, jamais une vue d'une entrée (même quand il lui est égal) ; une entrée qui contient un NaN lève `ValueError`.

Écris `reparameterize(mu, log_var, rng=None)` et `reparameterize_backward(dz, eps, log_var)`. **Contrat** propre à ces deux fonctions :
- `mu`, `log_var`, `dz` et `eps` sont des tableaux 2-D `(n, d)` de **même forme** (pas de diffusion : `(2, 3)` face à `(1, 3)` est une `ValueError`) ;
- `rng` accepte `None`, un entier (une graine) ou un `np.random.Generator` ; la même graine donne les mêmes tirages ; `eps` est tiré par `standard_normal` du générateur, et la fonction rend `(z, eps)` ;
- `reparameterize_backward` rend `(dmu, dlog_var)`, les gradients de la loss par rapport à `mu` et `log_var` quand on connaît `dz` (le ∂ 25.5 les a dérivés) ; `dmu` est un nouveau tableau, même s'il est égal à `dz`.

La première cellule de vérification essaie tes fonctions : `z` contre sa formule, la reproductibilité, la moyenne et la variance de 200 000 tirages, et la passe arrière contre `torch.autograd`. La seconde lance les tests (`-k "test_reparameterize_"`).

Dans tes notes : pourquoi faut-il rendre `eps` ? Pourquoi le gradient par rapport à `log_var` est-il nul quand `eps` vaut 0, et que devient-il quand `log_var` est très négatif ?

In [ ]:
# Nothing to write here: reparameterize and reparameterize_backward go in mon_travail/mylearn/vae.py

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="25")   # reload your saved files
with checked("25.28"):
    V = mylearn.vae
    mu_28 = np.array([[0.5, -1.0, 2.0], [0.0, 3.0, -0.5]])
    lv_28 = np.array([[0.0, 1.0, -2.0], [0.5, -1.0, 2.0]])
    z_28, eps_28 = V.reparameterize(mu_28, lv_28, rng=np.random.default_rng(0))
    same("25.28", "z rendu par reparameterize : mu + exp(0.5 log_var) * eps, avec son eps",
         z_28, mu_28 + np.exp(0.5 * lv_28) * np.asarray(eps_28))
    same("25.28", "la même graine donne le même eps", V.reparameterize(mu_28, lv_28, rng=0)[1],
         V.reparameterize(mu_28, lv_28, rng=0)[1])
    big_28 = V.reparameterize(np.full((200_000, 1), 2.0), np.full((200_000, 1), np.log(9.0)), rng=1)[0]
    verdict("25.28", abs(float(np.mean(big_28)) - 2.0) < 0.03 and abs(float(np.var(big_28)) - 9.0) < 0.15,
            f"200 000 tirages de N(2, 9) : moyenne {fr(float(np.mean(big_28)))}, variance {fr(float(np.var(big_28)))}.",
            "200 000 tirages avec mu = 2 et log_var = ln 9 doivent avoir une moyenne proche de 2 et une variance proche "
            f"de 9 : reçu {float(np.mean(big_28)):.3f} et {float(np.var(big_28)):.3f}.")
    dz_28 = np.array([[1.0, -2.0, 0.5], [0.3, 0.0, -1.0]])
    mu_t = torch.tensor(mu_28, requires_grad=True)
    lv_t = torch.tensor(lv_28, requires_grad=True)
    (torch.tensor(dz_28) * (mu_t + torch.exp(0.5 * lv_t) * torch.tensor(np.asarray(eps_28)))).sum().backward()
    dmu_28, dlv_28 = V.reparameterize_backward(dz_28, eps_28, lv_28)
    same("25.28", "dmu de reparameterize_backward, contre torch.autograd", dmu_28, mu_t.grad.numpy())
    same("25.28", "dlog_var de reparameterize_backward, contre torch.autograd", dlv_28, lv_t.grad.numpy())

In [ ]:
if written(lambda: mylearn.vae.reparameterize([[0.0]], [[0.0]], rng=0)):
    run_pytest_group(TEST_VAE, "test_reparameterize_")
else:
    print("⏳ Ex 25.28 : les tests attendent tes fonctions.")

### Ex 25.29 — Divergence KL gaussienne et son gradient 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire la KL d'un code gaussien au prior $\mathcal{N}(\mathbf{0}, \mathbf{I})$ et son gradient dans ta librairie.  
**Prérequis :** ∂ 25.7 · ch. 6 (KL) · fiche §25.9.1, §25.9.2 · **mylearn :** `vae.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/vae.py` (créé par `python tools/start_chapter.py 25`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur quelques cas ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_ch25_vae.py -q` les lance tous dans un terminal.

**Contrat commun de `vae.py`** (les docstrings, plus ces précisions que vérifient les tests) : les entrées peuvent être des listes et ne sont jamais modifiées ; chaque tableau rendu est un **nouveau** tableau `float64`, jamais une vue d'une entrée (même quand il lui est égal) ; une entrée qui contient un NaN lève `ValueError`.

Écris `gaussian_kl(mu, log_var)` et `gaussian_kl_grad(mu, log_var)`. **Contrat** propre à ces deux fonctions :
- `mu` et `log_var` sont des tableaux 2-D `(n, d)` de même forme, sans diffusion ;
- `gaussian_kl` rend un tableau `(n,)` : pour chaque exemple, la KL de $\mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ à $\mathcal{N}(\mathbf{0}, \mathbf{I})$, **sommée** sur les $d$ latents, en nats (la formule du ∂ 25.7) ;
- `gaussian_kl_grad` rend `(dmu, dlog_var)`, les dérivées de la KL de chaque exemple par rapport à `mu` et à `log_var`, de forme `(n, d)` ; `dmu` est un nouveau tableau, même s'il est égal à `mu` ;
- pour des `log_var` proches de 0, préfère `np.expm1(log_var)` à `np.exp(log_var) - 1` (moins d'erreur d'arrondi).

La première cellule de vérification compare ta KL à `torch.distributions.kl_divergence` et ton gradient à `torch.autograd`. La seconde lance les tests (`-k "test_gaussian_kl_"`).

Dans tes notes : en quels `mu` et `log_var` la KL est-elle minimale, et que vaut-elle là ? Pourquoi somme-t-on sur les latents au lieu de moyenner ?

In [ ]:
# Nothing to write here: gaussian_kl and gaussian_kl_grad go in mon_travail/mylearn/vae.py

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="25")   # reload your saved files
with checked("25.29"):
    V = mylearn.vae
    mu_29 = np.array([[0.5, -1.0, 2.0], [0.0, 3.0, -0.5]])
    lv_29 = np.array([[0.0, 1.0, -2.0], [0.5, -1.0, 2.0]])
    mu_t = torch.tensor(mu_29, requires_grad=True)
    lv_t = torch.tensor(lv_29, requires_grad=True)
    kl_t = torch.distributions.kl_divergence(torch.distributions.Normal(mu_t, torch.exp(0.5 * lv_t)),
                                             torch.distributions.Normal(0.0, 1.0)).sum(dim=1)
    same("25.29", "gaussian_kl, contre torch.distributions.kl_divergence (une valeur par exemple)",
         V.gaussian_kl(mu_29, lv_29), kl_t.detach().numpy())
    kl_t.sum().backward()
    gmu_29, glv_29 = V.gaussian_kl_grad(mu_29, lv_29)
    same("25.29", "gaussian_kl_grad : d KL / d mu, contre torch.autograd", gmu_29, mu_t.grad.numpy())
    same("25.29", "gaussian_kl_grad : d KL / d log_var, contre torch.autograd", glv_29, lv_t.grad.numpy())
    same("25.29", "la KL de N(0, 1) contre elle-même", V.gaussian_kl(np.zeros((1, 4)), np.zeros((1, 4))), [0.0])

In [ ]:
if written(lambda: mylearn.vae.gaussian_kl([[0.0]], [[0.0]])):
    run_pytest_group(TEST_VAE, "test_gaussian_kl_")
else:
    print("⏳ Ex 25.29 : les tests attendent tes fonctions.")

### Ex 25.30 — Un VAE en PyTorch : reconstruction + KL 📦 ★★ ⏱️ 30 min
**Objectif :** écrire un VAE en PyTorch et sa loss, la reconstruction plus $\beta$ fois la KL.  
**Prérequis :** Ex 25.28 · Ex 25.29 · Ex 25.16 · fiche §25.9.2, §25.10 · **Fil rouge :** MNIST · **Parcours :** R, M, C

Ce que tu as écrit en NumPy, PyTorch le dérive tout seul. Le VAE de cet exercice suit celui de la fiche : un encodeur dense qui rend, pour chaque latent, une moyenne et une log-variance ; un tirage par la reparamétrisation ; un décodeur qui rend des **logits** (la sigmoïde est dans la loss, plus stable).

**Contrat**
- `VAE30(n_latent=20, hidden=(512, 256))` est un `nn.Module` : `encoder` = `nn.Sequential(Linear(784, hidden[0]), ReLU, Linear(hidden[0], hidden[1]), ReLU, …)` ; `to_mu` et `to_log_var`, deux `nn.Linear(hidden[-1], n_latent)` appliqués à la sortie de l'encodeur ; `decoder` = `nn.Sequential(Linear(n_latent, hidden[-1]), ReLU, …, Linear(hidden[0], 784))`, sans sigmoïde ; les attributs `n_latent` et `hidden` (un tuple) gardent les deux arguments ;
- `encode(x)` rend `(mu, log_var)` ; `forward(x)` rend `(logits, mu, log_var)`, avec `z = mu + exp(0.5 log_var) * eps` et `eps = torch.randn_like(mu)` ;
- `vae_loss_30(x, logits, mu, log_var, beta=1.0)` rend `(loss, rec, kl)`, trois tenseurs : `rec` est la cross-entropy binaire calculée **sur les logits**, **sommée** sur les 784 pixels puis moyennée sur le batch ; `kl` la KL au prior, sommée sur les latents puis moyennée sur le batch ; `loss = rec + beta * kl`.

La vérification regarde les couches, puis la sortie de `forward` (la même graine, la même sortie), et compare ta loss à un calcul fait avec `torch.distributions`. La cellule suivante donne les versions de référence (`VAE`, `vae_loss`, `train_vae`, `vae_eval`, `vae_images`) et entraîne `VAE_C`, le VAE que la suite explore ; la dernière entraîne **ton** VAE avec ta loss (3 epochs en mode rapide) et décode des codes tirés de $\mathcal{N}(\mathbf{0}, \mathbf{I})$.

Dans tes notes : pourquoi sommer la reconstruction sur les pixels (fiche, pièges) ? Que vaut la KL de ton VAE après 3 epochs, en nats, et que dirait-elle en bits ?

In [ ]:
class VAE30(nn.Module):
    """A VAE for rows of 784 pixels.
    encoder: nn.Sequential(Linear(784, hidden[0]), ReLU, Linear(hidden[0], hidden[1]), ReLU, ...);
    to_mu and to_log_var: two nn.Linear(hidden[-1], n_latent) on the output of the encoder;
    decoder: nn.Sequential(Linear(n_latent, hidden[-1]), ReLU, ..., Linear(hidden[0], 784)): LOGITS, no sigmoid.
    Attributes n_latent and hidden (a tuple) keep the two arguments."""

    def __init__(self, n_latent=20, hidden=(512, 256)):
        super().__init__()
        raise NotImplementedError("VAE30.__init__")

    def encode(self, x):
        """(mu, log_var), both (N, n_latent)."""
        raise NotImplementedError("VAE30.encode")

    def forward(self, x):
        """(logits, mu, log_var): z = mu + exp(0.5 log_var) * eps with eps = torch.randn_like(mu), decoded."""
        raise NotImplementedError("VAE30.forward")


def vae_loss_30(x, logits, mu, log_var, beta=1.0):
    """(loss, rec, kl), three tensors: rec = the binary cross-entropy (from the logits) SUMMED over the 784 pixels,
    then averaged over the batch; kl = the KL of N(mu, exp(log_var)) from N(0, I), summed over the latents, then
    averaged over the batch; loss = rec + beta * kl."""
    raise NotImplementedError("vae_loss_30")

In [ ]:
with checked("25.30"):
    torch.manual_seed(0)
    vae_30 = VAE30(20)
    if returned("25.30", "VAE30(20)", vae_30):
        want_30 = (["Linear(784, 512)", "ReLU", "Linear(512, 256)", "ReLU"], "Linear(256, 20)",
                   ["Linear(20, 256)", "ReLU", "Linear(256, 512)", "ReLU", "Linear(512, 784)"])
        got_30 = (layout(vae_30.encoder), layout([vae_30.to_mu])[0], layout(vae_30.decoder))
        verdict("25.30", got_30 == want_30 and layout([vae_30.to_log_var]) == ["Linear(256, 20)"]
                and vae_30.n_latent == 20 and tuple(vae_30.hidden) == (512, 256),
                "VAE30(20) : encodeur, to_mu, to_log_var et décodeur (des logits) suivent le contrat.",
                f"attendu {want_30} (et to_log_var comme to_mu), reçu {got_30}.")
        x_30 = to_tensor(X_MNIST[:5])
        torch.manual_seed(1)
        logits_30, mu_30, lv_30 = vae_30(x_30)
        torch.manual_seed(1)
        again_30 = vae_30(x_30)[0]
        mu_ref_30, lv_ref_30 = vae_30.encode(x_30)
        torch.manual_seed(1)
        want_logits_30 = vae_30.decoder(mu_ref_30 + torch.exp(0.5 * lv_ref_30) * torch.randn_like(mu_ref_30))
        verdict("25.30", tuple(logits_30.shape) == (5, 784) and tuple(mu_30.shape) == (5, 20)
                and torch.equal(logits_30, again_30) and not torch.equal(logits_30, vae_30(x_30)[0])
                and torch.equal(mu_30, mu_ref_30) and torch.equal(lv_30, lv_ref_30)
                and torch.allclose(logits_30, want_logits_30, atol=1e-6),
                "forward rend (logits, mu, log_var), tire un code neuf à chaque appel (torch.randn_like).",
                "forward doit rendre (logits (5, 784), mu, log_var) avec mu, log_var = encode(x), décoder "
                "z = mu + exp(0.5 log_var) * eps et tirer eps par torch.randn_like(mu) : la même graine, la même sortie ; "
                "un autre appel, une autre sortie.")
    g_30 = torch.Generator().manual_seed(30)
    x_t = torch.rand(6, 784, generator=g_30)
    logits_t, mu_t, lv_t = (torch.randn(6, 784, generator=g_30), torch.randn(6, 4, generator=g_30, requires_grad=True),
                            torch.randn(6, 4, generator=g_30))
    result_30 = vae_loss_30(x_t, logits_t, mu_t, lv_t, beta=2.0)
    if returned("25.30", "vae_loss_30", result_30):
        rec_ref_30 = F.binary_cross_entropy_with_logits(logits_t, x_t, reduction="sum") / 6     # summed, then / batch
        kl_ref_30 = torch.distributions.kl_divergence(torch.distributions.Normal(mu_t, torch.exp(0.5 * lv_t)),
                                                      torch.distributions.Normal(0.0, 1.0)).sum(dim=1).mean()
        same("25.30", "vae_loss_30(..., beta=2) : (loss, rec, kl)", [float(v.detach()) for v in result_30],
             [float((rec_ref_30 + 2 * kl_ref_30).detach()), float(rec_ref_30), float(kl_ref_30.detach())], rtol=1e-5)
        result_30[0].backward()
        verdict("25.30", mu_t.grad is not None and float(mu_t.grad.abs().sum()) > 0,
                "la loss se dérive par rapport à mu.", "la loss doit rester un tenseur dérivable (pas de .item()).")

In [ ]:
# The reference VAE (your 25.30), its loss, its training loop, and VAE_C: the VAE that part C explores,
# trained the same way in both modes (the first N_FIXED images, 10 epochs, seed 2530; about 15 s on a CPU)
def make_vae_decoder(n_latent, hidden=(512, 256)):
    """The decoder of the VAE: Linear(n_latent, hidden[-1]) + ReLU, ..., Linear(hidden[0], 784) (logits)."""
    layers, n_in = [], n_latent
    for n_out in reversed(hidden):
        layers += [nn.Linear(n_in, n_out), nn.ReLU()]
        n_in = n_out
    return nn.Sequential(*layers, nn.Linear(n_in, 784))


class VAE(nn.Module):
    """The VAE of 25.30: encoder (dense ReLU layers), to_mu and to_log_var, decoder (logits)."""

    def __init__(self, n_latent=20, hidden=(512, 256)):
        super().__init__()
        self.n_latent, self.hidden = n_latent, tuple(hidden)
        layers, n_in = [], 784
        for n_out in hidden:
            layers += [nn.Linear(n_in, n_out), nn.ReLU()]
            n_in = n_out
        self.encoder = nn.Sequential(*layers)
        self.to_mu, self.to_log_var = nn.Linear(n_in, n_latent), nn.Linear(n_in, n_latent)
        self.decoder = make_vae_decoder(n_latent, hidden)

    def encode(self, x):
        h = self.encoder(x)
        return self.to_mu(h), self.to_log_var(h)

    def forward(self, x):
        mu, log_var = self.encode(x)
        return self.decoder(mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)), mu, log_var


def vae_loss(x, logits, mu, log_var, beta=1.0):
    """(loss, rec, kl): the binary cross-entropy summed over the pixels, the KL summed over the latents, both
    averaged over the batch; loss = rec + beta * kl."""
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1).mean()
    kl = 0.5 * (mu ** 2 + torch.exp(log_var) - 1 - log_var).sum(dim=1).mean()
    return rec + beta * kl, rec, kl


def vae_eval(model, x, batch_size=1000):
    """(rec, kl, kl_per_latent) of the images x: rec decoded from the means mu (no draw), kl exact, both averaged
    over the images; kl_per_latent: the mean KL of each latent, an array (n_latent,)."""
    model.eval()
    rec, kls = 0.0, []
    x = to_tensor(x)
    with torch.inference_mode():
        for i in range(0, len(x), batch_size):
            xb = x[i:i + batch_size].to(DEVICE)
            mu, log_var = model.encode(xb)
            rec += F.binary_cross_entropy_with_logits(model.decoder(mu), xb, reduction="sum").item()
            kls.append((0.5 * (mu ** 2 + torch.exp(log_var) - 1 - log_var)).cpu())
    per_latent = torch.cat(kls).mean(dim=0).numpy()
    return rec / len(x), float(per_latent.sum()), per_latent


def train_vae(model, x, epochs, beta=1.0, lr=1e-3, batch_size=128, seed=0, loss_fn=None, verbose=False):
    """Train the VAE `model` in place (on DEVICE) on the images x (N, 784): Adam(lr), mini-batches in an order drawn
    from `seed`, the draws of the codes from torch.manual_seed(seed), loss_fn(x, logits, mu, log_var, beta) (vae_loss
    by default). Returns the history: "loss", "rec", "kl" (means of each epoch), "seconds"."""
    torch.manual_seed(seed)
    model.to(DEVICE)
    loss_fn = loss_fn or vae_loss
    x = to_tensor(x)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    history = {"loss": [], "rec": [], "kl": [], "seconds": []}
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        totals = np.zeros(3)
        order = torch.randperm(len(x), generator=g)
        for i in range(0, len(x), batch_size):
            xb = x[order[i:i + batch_size]].to(DEVICE)
            loss, rec, kl = loss_fn(xb, *model(xb), beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            totals += np.array([loss.item(), rec.item(), kl.item()]) * len(xb)
        for key, value in zip(("loss", "rec", "kl"), totals / len(x)):
            history[key].append(float(value))
        history["seconds"].append(time.perf_counter() - start)
        if verbose:
            print(f"epoch {epoch:2d} · loss {history['loss'][-1]:.1f} · rec {history['rec'][-1]:.1f} · kl {history['kl'][-1]:.1f}")
    return history


def vae_images(model, z):
    """The images (N, 784) that the decoder of `model` draws from the codes z: sigmoid of the logits."""
    model.eval()
    return 1 / (1 + np.exp(-decode(model, np.asarray(z, dtype=np.float32))))


torch.manual_seed(2530)
VAE_C = VAE(20)
train_vae(VAE_C, X_MNIST[:N_FIXED], 10, seed=2530)
rec_c, kl_c, _ = vae_eval(VAE_C, XM_TEST)
print(f"VAE_C : reconstruction {rec_c:.1f} nats et KL {kl_c:.1f} nats par image de test")

In [ ]:
def encode_mu(model, x):
    """The means mu of the codes of the images x, for a VAE (model.encode): a NumPy array (eval mode, no graph)."""
    model.eval()
    return _batched(lambda xb: model.encode(xb)[0], x)

In [ ]:
with checked("25.30"):
    EPOCHS_30 = 3 if FAST_MODE else 20
    torch.manual_seed(2530)
    mine_30 = VAE30(20)
    history_30 = train_vae(mine_30, X_MNIST, EPOCHS_30, seed=2530, loss_fn=vae_loss_30, verbose=True)
    z_30 = np.random.default_rng(30).standard_normal((10, 20))
    show_rows([XM_TEST[:10], vae_images(mine_30, encode_mu(mine_30, XM_TEST[:10])), vae_images(mine_30, z_30)],
              ["originaux", "décodés de mu", "codes tirés de N(0, I)"], f"ton VAE, {EPOCHS_30} epochs")

### Ex 25.31 — Prédire : décoder des tirages N(0, I) avec un AE puis avec un VAE 🔮 ★★ ⏱️ 15 min
**Objectif :** prédire, puis mesurer, ce que deviennent des codes tirés au hasard avec un autoencodeur classique et avec un VAE.  
**Prérequis :** Ex 25.30 · Ex 25.19 · fiche §25.9, §25.10 · **Fil rouge :** MNIST · **Parcours :** R, C

La fiche le dit : des codes tirés au hasard donnent des taches avec un autoencodeur classique, et des chiffres avec un VAE. Combien ? Les mêmes 1 000 codes tirés de $\mathcal{N}(\mathbf{0}, \mathbf{I})$ (`Z_31`, 20 nombres chacun) passent dans le décodeur de `AE_DEEP` et dans celui de `VAE_C`, puis un petit classifieur de chiffres (`CLF_31`, entraîné par la cellule donnée) dit quel chiffre il voit dans chaque image, et avec quelle probabilité.

**Avant d'exécuter**, écris tes trois prédictions (`True` ou `False`) :
a) pour `AE_DEEP`, le chiffre le plus souvent prédit couvre-t-il **plus de 20 %** des 1 000 images (deux fois la part d'un chiffre sur dix) ?
b) pour `VAE_C`, la même question ;
c) **plus de la moitié** des images de `VAE_C` sont-elles reconnues avec confiance (probabilité > 0,9) ?

In [ ]:
# CLF_31: a small digit classifier (an MLP trained 5 epochs on the first N_FIXED images), and 1 000 codes drawn from N(0, I)
torch.manual_seed(2531)
CLF_31 = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(DEVICE)
_opt_31 = torch.optim.Adam(CLF_31.parameters(), lr=1e-3)
_x_31, _y_31 = to_tensor(X_MNIST[:N_FIXED]), torch.from_numpy(Y_MNIST[:N_FIXED])
_g_31 = torch.Generator().manual_seed(2531)
for _ in range(5):
    _order_31 = torch.randperm(N_FIXED, generator=_g_31)
    for _i in range(0, N_FIXED, 128):
        _idx = _order_31[_i:_i + 128]
        _loss = F.cross_entropy(CLF_31(_x_31[_idx].to(DEVICE)), _y_31[_idx].to(DEVICE))
        _opt_31.zero_grad()
        _loss.backward()
        _opt_31.step()


def classify_31(images):
    """(predicted digit, probability of that digit) for each image, two NumPy arrays."""
    probs = torch.softmax(torch.from_numpy(_batched(CLF_31, images)), dim=1).numpy()
    return probs.argmax(axis=1), probs.max(axis=1)


Z_31 = np.random.default_rng(2531).standard_normal((1000, 20)).astype(np.float32)
print(f"CLF_31 : accuracy de test {np.mean(classify_31(XM_TEST)[0] == YM_TEST):.3f}")

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
ae_top_31 = ...    # a) the codes Z_31 decoded by AE_DEEP: does the digit the classifier predicts most often cover
                   #    more than 20 % of the 1 000 images? True or False
vae_top_31 = ...   # b) the same question for the codes decoded by VAE_C: True or False
vae_conf_31 = ...  # c) are more than half of the VAE_C images recognised with confidence (probability > 0.9)?
                   #    True or False

In [ ]:
if any(answer is ... or answer is None for answer in [ae_top_31, vae_top_31, vae_conf_31]):
    print("⏳ Ex 25.31 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    with checked("25.31"):
        images_31 = {"AE_DEEP": decode(AE_DEEP, Z_31), "VAE_C": vae_images(VAE_C, Z_31)}
        for name_31, imgs_31 in images_31.items():
            pred_31, conf_31 = classify_31(imgs_31)
            counts_31 = np.bincount(pred_31, minlength=10) / len(pred_31)
            print(f"{name_31:8s} chiffre le plus prédit : {counts_31.argmax()} ({counts_31.max():.0%} des tirages) · "
                  f"reconnus avec confiance (p > 0,9) : {np.mean(conf_31 > 0.9):.0%} · répartition : "
                  + " ".join(f"{c:.0%}" for c in counts_31))
        codes_ae_31 = encode(AE_DEEP, XM_TEST)
        print(f"normes des codes : tirages de N(0, I) {np.linalg.norm(Z_31, axis=1).mean():.1f} · codes de AE_DEEP "
              f"{np.linalg.norm(codes_ae_31, axis=1).mean():.1f} · moyennes mu de VAE_C {np.linalg.norm(encode_mu(VAE_C, XM_TEST), axis=1).mean():.1f}")
        show_rows([images_31["AE_DEEP"][:12], images_31["VAE_C"][:12]], ["AE_DEEP", "VAE_C"], "les mêmes codes, deux décodeurs")
    wb.check("25.31a", ae_top_31)
    wb.check("25.31b", vae_top_31)
    wb.check("25.31c", vae_conf_31)

Dans tes notes : que t'apprend la répartition des chiffres prédits sur ce que les tirages rencontrent dans l'espace latent de `AE_DEEP` (compare les normes affichées) ? Pourquoi un classifieur peut-il être « sûr de lui » sur une tache ? Qu'est-ce qui limite la netteté des images du VAE ici (25.36) ?

### Ex 25.32 — Un VAE est stochastique : sorties répétées, bruit relatif, interpolations 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer ce que le tirage du code change aux sorties d'un VAE, et sa tolérance au bruit face à un autoencodeur.  
**Prérequis :** Ex 25.30 · Ex 25.31 · fiche §25.10 · **Fil rouge :** MNIST · **Parcours :** C

Le livre passe huit fois le même 2 dans son VAE et obtient huit sorties voisines ; il bruite les codes de 10 % puis de 30 % de leur valeur. Tu refais ces expériences, en comparant `VAE_C` à `AE_DEEP`, et tu mesures avec `classify_31` (25.31) la part des sorties qui restent des chiffres nets.

**Contrat**
- `repeat_32(vae, x, k, seed=0)` passe $k$ fois **une** image `x` (784 pixels) dans le VAE : `mu, log_var = vae.encode(x)` (une ligne), puis, après `torch.manual_seed(seed)`, `z = mu + exp(0.5 log_var) * torch.randn(k, n_latent)`, décodé (la sigmoïde des logits), en mode évaluation et sans graphe : un tableau NumPy `(k, 784)` ;
- `relative_noise_32(model, mu, frac, seed=0)` rend les codes `mu` `(N, d)` bruités de `frac` de leur propre valeur : $\mu \,(1 + \text{frac} \cdot u)$, avec $u$ uniforme dans $[-1, 1]$ tiré par `np.random.default_rng(seed).uniform(-1, 1, size=mu.shape)` ; un nouveau tableau `float32` (l'argument `model` ne sert qu'à garder la même signature pour les deux modèles).

La vérification compare tes fonctions à des calculs faits avec les mêmes graines. La cellule suivante montre huit passages d'un 2, compare `VAE_C` et `AE_DEEP` face à des bruits de 10 %, 30 % et 100 %, et interpole d'un 4 à un 9 avec ta `interpolate_latents`.

Dans tes notes : d'où vient la variété des huit sorties ? Lequel des deux modèles tolère le mieux un code bruité, et pourquoi (pense à ce que la KL impose aux codes) ? L'interpolation du VAE passe-t-elle par des chiffres à chaque étape ?

In [ ]:
def repeat_32(vae, x, k, seed=0):
    """k passes of ONE image x (784 pixels) through the VAE: mu, log_var = vae.encode(x), then after
    torch.manual_seed(seed), z = mu + exp(0.5 log_var) * torch.randn(k, n_latent), decoded (sigmoid of the logits),
    in eval mode and without a graph. A NumPy array (k, 784)."""
    raise NotImplementedError("repeat_32")


def relative_noise_32(model, mu, frac, seed=0):
    """The codes mu (N, d) with a noise of `frac` of their own value: mu * (1 + frac * u), u uniform in [-1, 1],
    drawn by np.random.default_rng(seed).uniform(-1, 1, size=mu.shape); a new float32 array."""
    raise NotImplementedError("relative_noise_32")

In [ ]:
with checked("25.32"):
    two_32 = XM_TEST[np.flatnonzero(YM_TEST == 2)[0]]
    rep_32 = repeat_32(VAE_C, two_32, 8, seed=5)
    if returned("25.32", "repeat_32", rep_32):
        mu_32, lv_32 = (t.detach() for t in VAE_C.encode(to_tensor(two_32)[None].to(DEVICE)))
        torch.manual_seed(5)
        want_32 = torch.sigmoid(VAE_C.decoder(mu_32 + torch.exp(0.5 * lv_32) * torch.randn(8, 20).to(DEVICE))).detach().cpu().numpy()
        same("25.32", "repeat_32(VAE_C, un 2, 8, seed=5) : huit tirages du même code", rep_32, want_32, rtol=1e-4, atol=1e-6)
    noisy_32 = relative_noise_32(VAE_C, np.array([[1.0, -2.0], [0.0, 4.0]]), 0.1, seed=3)
    if returned("25.32", "relative_noise_32", noisy_32):
        same("25.32", "relative_noise_32([[1, -2], [0, 4]], 0.1, seed=3)", noisy_32,
             np.array([[1.0, -2.0], [0.0, 4.0]]) * (1 + 0.1 * np.random.default_rng(3).uniform(-1, 1, size=(2, 2))),
             rtol=1e-6)

In [ ]:
with checked("25.32"):
    two_32 = XM_TEST[np.flatnonzero(YM_TEST == 2)[0]]
    rep_32 = repeat_32(VAE_C, two_32, 8, seed=32)
    print(f"huit passages du même 2 : écart-type moyen d'un pixel d'un passage à l'autre {rep_32.std(axis=0).mean():.3f}")
    show_rows([np.vstack([two_32[None], rep_32])], ["un 2, puis 8 passages"])
    mu_32 = encode_mu(VAE_C, XM_TEST[:500])
    codes_32 = encode(AE_DEEP, XM_TEST[:500])
    for frac_32 in (0.1, 0.3, 1.0):
        conf_vae_32 = np.mean(classify_31(vae_images(VAE_C, relative_noise_32(VAE_C, mu_32, frac_32, seed=32)))[1] > 0.9)
        conf_ae_32 = np.mean(classify_31(decode(AE_DEEP, relative_noise_32(AE_DEEP, codes_32, frac_32, seed=32)))[1] > 0.9)
        print(f"bruit de {frac_32:.0%} de la valeur des codes : reconnus avec confiance · VAE_C {conf_vae_32:.0%} · AE_DEEP {conf_ae_32:.0%}")
    show_rows([vae_images(VAE_C, relative_noise_32(VAE_C, mu_32[:10], f, seed=32)) for f in (0.0, 0.1, 0.3, 1.0)],
              ["0 %", "10 %", "30 %", "100 %"], "VAE_C : les codes de 10 chiffres, bruités")
    with_mylearn_32 = written(lambda: mylearn.vae.interpolate_latents([0.0], [1.0], 2))
    if with_mylearn_32:
        a_32, b_32 = encode_mu(VAE_C, XM_TEST[[np.flatnonzero(YM_TEST == 4)[0], np.flatnonzero(YM_TEST == 9)[1]]])
        show_rows([vae_images(VAE_C, mylearn.vae.interpolate_latents(a_32, b_32, 8, method="slerp"))], ["VAE_C, slerp"],
                  "d'un 4 à un 9")
    else:
        print("⏳ Ex 25.32 : l'interpolation attend ta fonction interpolate_latents (25.20).")

### Ex 25.33 — Reproduire le nuage latent d'un VAE 2D et la grille −3…3 (fig. 25.67-25.68) 🎨 ★★ ⏱️ 30 min
**Objectif :** dessiner l'espace latent d'un VAE à 2 latents et le comparer au prior.  
**Prérequis :** Ex 25.30 · Ex 25.18 · fiche §25.9.1, §25.10 · **Fil rouge :** MNIST · **Parcours :** C

Le livre dessine les codes d'un VAE à 2 latents (fig. 25.67), puis décode une grille de codes de −3 à 3 sur chaque axe (fig. 25.68). La cellule donnée entraîne `VAE_2D` et donne `grid_codes`, la version de référence de ta `grid_codes_18`.

**Contrat** : `plot_vae_map_33(vae, x, labels, lim=3.0, n=15)` trace deux panneaux côte à côte : à gauche, les moyennes $\boldsymbol{\mu}$ des codes de `x` (`encode_mu`), colorées par chiffre, avec une légende, les cercles de rayon 1, 2 et 3 autour de l'origine, et les limites $-$`lim` à `lim` sur les deux axes ; à droite, les $n \times n$ images que `vae_images` tire de `grid_codes((-lim, lim, -lim, lim), n)`, en **une** image de $28n \times 28n$ pixels, le code du coin en haut à gauche en premier ; elle finit par `plt.show()` et rend la mosaïque.

La vérification compare le coin de ta mosaïque au décodage de $(-3, 3)$, puis affiche la part des $\boldsymbol{\mu}$ à moins de 1, 2 et 3 de l'origine, face à celle de tirages de $\mathcal{N}(\mathbf{0}, \mathbf{I})$.

Dans tes notes : compare ce nuage à la carte de 25.18 : où sont les codes, reste-t-il des zones vides ? La grille se décode-t-elle en chiffres « presque partout », comme le dit la fiche ? Pourquoi les $\boldsymbol{\mu}$ seuls ne remplissent-ils pas tout à fait la gaussienne ?

In [ ]:
# VAE_2D: the VAE of 25.30 with a code of 2 numbers (the first N_FIXED images, 10 epochs, ≈ 15 s), and grid_codes,
# the reference version of grid_codes_18
torch.manual_seed(2533)
VAE_2D = VAE(2)
train_vae(VAE_2D, X_MNIST[:N_FIXED], 10, seed=2533)


def grid_codes(lims, n):
    """The n × n codes of a regular grid over lims = (xmin, xmax, ymin, ymax), row by row from the top (25.18)."""
    xmin, xmax, ymin, ymax = lims
    gx, gy = np.meshgrid(np.linspace(xmin, xmax, n), np.linspace(ymax, ymin, n))
    return np.column_stack([gx.ravel(), gy.ravel()])


print(f"VAE_2D : reconstruction {vae_eval(VAE_2D, XM_TEST)[0]:.1f} nats par image de test")

In [ ]:
def plot_vae_map_33(vae, x, labels, lim=3.0, n=15):
    """Two panels side by side: (left) the means mu of the codes of x (encode_mu), coloured by digit, with a legend,
    the circles of radius 1, 2 and 3 around the origin, and the limits -lim to lim on both axes; (right) the
    n × n images that vae_images draws from grid_codes((-lim, lim, -lim, lim), n), in ONE image of (28 n) × (28 n)
    pixels, the code of the top-left corner first. Ends with plt.show(). Returns the mosaic."""
    raise NotImplementedError("plot_vae_map_33")

In [ ]:
with checked("25.33"):
    mosaic_33 = plot_vae_map_33(VAE_2D, XM_TEST, YM_TEST)
    if returned("25.33", "plot_vae_map_33", mosaic_33):
        corner_33 = vae_images(VAE_2D, [[-3.0, 3.0]]).reshape(28, 28)
        verdict("25.33", np.shape(mosaic_33) == (420, 420) and np.allclose(np.asarray(mosaic_33)[:28, :28], corner_33, atol=1e-5),
                "la mosaïque (420, 420) commence par le code (-3, 3).",
                f"attendu une mosaïque (420, 420) dont l'image en haut à gauche décode (-3, 3), reçu {shape_text(mosaic_33)}.")
    radius_33 = np.linalg.norm(encode_mu(VAE_2D, XM_TEST), axis=1)
    print(f"part des moyennes mu à moins de 1, 2 et 3 de l'origine : {np.mean(radius_33 < 1):.0%}, "
          f"{np.mean(radius_33 < 2):.0%}, {np.mean(radius_33 < 3):.0%} (pour des tirages de N(0, I) : 39 %, 86 %, 99 %)")

### Ex 25.34 — Diagnostiquer des courbes reconstruction/KL (effondrement du posterior) 📈 ★★ ⏱️ 15 min
**Objectif :** reconnaître, sur des courbes réelles, l'effondrement du posterior, un $\beta$ trop fort et une montée progressive de $\beta$.  
**Prérequis :** Ex 25.30 · fiche (Au-delà du livre : β-VAE, effondrement du posterior) · **Fil rouge :** MNIST · **Parcours :** R, M, C

`HISTORY_34`, calculé à l'avance par `tools/chapters/history_ch25.py`, contient quatre entraînements du même VAE (20 latents, 20 epochs sur 20 000 chiffres), qui ne diffèrent que par $\beta$ : l'un garde $\beta = 1$, un autre $\beta = 20$, un autre $\beta = 100$, et le dernier fait monter $\beta$ de 0 à 1 pendant les 10 premières epochs (le *KL annealing*). Tu ne sais pas lequel est lequel. Chaque run donne, epoch par epoch, la reconstruction et la KL de validation (en nats par image) et le nombre de **latents actifs** (dont la KL moyenne dépasse 0,05 nat). La cellule donnée trace les trois courbes.

Réponds en lisant les courbes, ou en calculant sur `HISTORY_34` (des listes, l'epoch 1 à l'indice 0) :
a) `collapsed_34` : la lettre du run dont le posterior s'est effondré (la KL presque nulle pour tous les latents) ;
b) `active_34` : dans le run qui finit avec 1 à 5 latents actifs, combien en a-t-il à la dernière epoch ?
c) `peak_34` : dans le run dont la KL part le plus haut, la KL de la première epoch moins celle de la dernière, en nats (1 décimale) ;
d) `gap_34` : la dernière reconstruction du run de b), moins la dernière reconstruction du run dont la KL finit la plus haute, en nats (1 décimale) ;
e) `mapping_34` : les lettres des runs à $\beta = 1$, $\beta = 20$, $\beta = 100$ et $\beta$ montant, dans cet ordre (par exemple `"ABCD"`).

In [ ]:
HISTORY_34 = {   # four trainings of the same VAE (20 latents, 20 epochs, MNIST), tools/chapters/history_ch25.py
    "A": {"rec": [158.1, 119.36, 105.62, 95.96, 90.24, 86.61, 83.55, 81.98, 81.45, 80.86, 80.16, 79.4, 78.44, 77.42, 77.67, 76.76, 76.52, 75.68, 75.6, 75.15],
          "kl": [68.01, 45.27, 35.98, 32.72, 30.64, 29.59, 27.4, 25.92, 23.81, 23.36, 23.16, 21.88, 22.95, 22.12, 22.1, 21.68, 22.09, 22.71, 22.24, 22.38],
          "active": [20, 20, 20, 20, 20, 20, 20, 20, 19, 19, 19, 19, 18, 18, 18, 18, 18, 17, 16, 16]},
    "B": {"rec": [192.58, 133.17, 116.83, 110.43, 103.66, 99.88, 96.26, 94.11, 92.19, 90.16, 88.85, 87.48, 86.62, 86.14, 85.37, 84.71, 83.89, 83.24, 82.54, 82.34],
          "kl": [5.33, 12.4, 13.87, 14.21, 14.96, 15.37, 15.69, 16.56, 16.55, 17.43, 17.24, 17.49, 17.72, 17.84, 18.1, 17.11, 18.35, 18.32, 18.64, 18.8],
          "active": [17, 20, 18, 14, 14, 13, 13, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]},
    "C": {"rec": [287.97, 252.09, 226.25, 213.69, 207.78, 206.63, 205.83, 205.53, 205.68, 205.69, 205.56, 205.91, 205.61, 205.87, 205.84, 205.53, 205.72, 205.81, 205.77, 205.66],
          "kl": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
          "active": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    "D": {"rec": [295.78, 272.67, 248.03, 228.17, 216.04, 208.88, 202.47, 200.66, 199.5, 196.42, 192.42, 191.01, 189.2, 190.07, 189.01, 188.83, 188.6, 187.96, 188.92, 188.94],
          "kl": [0.04, 0.05, 0.11, 0.16, 0.19, 0.16, 0.18, 0.17, 0.18, 0.27, 0.31, 0.32, 0.37, 0.36, 0.37, 0.41, 0.38, 0.42, 0.39, 0.41],
          "active": [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for name_34, run_34 in HISTORY_34.items():
    epochs_34 = range(1, len(run_34["rec"]) + 1)
    axes[0].plot(epochs_34, run_34["rec"], label=name_34)
    axes[1].plot(epochs_34, run_34["kl"], label=name_34)
    axes[2].plot(epochs_34, run_34["active"], label=name_34)
for ax, title in zip(axes, ("reconstruction (nats par image de validation)", "KL (nats par image)",
                            "latents actifs (KL moyenne > 0,05 nat)")):
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
collapsed_34 = ...  # a) the letter of the run whose posterior collapsed (KL almost 0 for all the latents)
active_34 = ...     # b) in the run that ends with 1 to 5 active latents, how many are active at the last epoch?
peak_34 = ...       # c) in the run whose KL starts the highest: its KL at the first epoch minus its KL at the
                    #    last epoch, in nats (1 decimal)
gap_34 = ...        # d) the last reconstruction of the run of b) minus the last reconstruction of the run whose
                    #    KL ends the highest, in nats (1 decimal)
mapping_34 = ...    # e) the letters of the runs for beta = 1, beta = 20, beta = 100 and beta rising from 0 to 1
                    #    over the first 10 epochs, in this order, e.g. "ABCD"

In [ ]:
wb.check("25.34a", collapsed_34)
wb.check("25.34b", active_34)
wb.check("25.34c", peak_34)
wb.check("25.34d", gap_34)
wb.check("25.34e", mapping_34)

Dans tes notes : à quoi reconnais-tu l'effondrement du posterior sur les trois courbes ? Que coûte un $\beta$ fort en reconstruction, et que gagne-t-on en échange (25.36) ? Pourquoi la KL du run à $\beta$ montant part-elle si haut, et pourquoi finit-il avec à la fois une meilleure reconstruction et plus de latents actifs que $\beta = 1$ ? Quelle courbe surveillerais-tu en entreprise pour repérer un effondrement tôt ?

### Ex 25.35 — Exporter le décodeur comme générateur autonome et le tester 🛠️ ★★ ⏱️ 20 min
**Objectif :** exporter le décodeur d'un VAE, le recharger sans le notebook, et le couvrir par des tests qui attrapent de vrais bugs.  
**Prérequis :** Ex 25.30 · ch. 23 (Ex 23.22, state_dict et rechargement) · fiche §25.10 · **Fil rouge :** MNIST · **Parcours :** R, C

Une fois le VAE entraîné, on jette l'encodeur et on garde le décodeur : c'est un **générateur**. Tu l'exportes dans un fichier qu'un autre programme peut recharger, sans rien de ce notebook, et tu écris deux tests ; la vérification s'assure qu'ils passent sur ton générateur **et** qu'ils échouent sur deux générateurs cassés (un test qui ne peut pas échouer ne vérifie rien).

**Contrat**
- `export_decoder_35(vae, path)` enregistre avec `torch.save`, dans le fichier `path`, le dictionnaire `{"n_latent": vae.n_latent, "hidden": list(vae.hidden), "state_dict": vae.decoder.state_dict()}` (le décodeur seul) ;
- `load_generator_35(path)` relit ce fichier (`torch.load(path, weights_only=True)`), reconstruit le décodeur (les couches de 25.30), charge ses poids, le passe en mode évaluation, et rend une fonction `generate(n, seed=0)` : `z = torch.randn(n, n_latent, generator=torch.Generator().manual_seed(seed))`, puis la sigmoïde des logits, × 255, arrondie (`torch.round`), en tableau NumPy `uint8` de forme `(n, 28, 28)` ; elle n'utilise que `torch`, `nn` et `numpy` ;
- tes tests `test_generator_shape_35()` (5 images de 28 × 28 en `uint8`) et `test_generator_seed_35()` (la même graine donne les mêmes images, deux graines différentes des images différentes) lisent le fichier par `GEN_PATH` ; ils tournent dans un autre processus, avec `wb.run_pytest`.

La vérification exporte le décodeur de `VAE_C`, recharge ton générateur et compare ses images à celles du décodeur, puis lance tes tests sur ton générateur et sur les deux générateurs cassés de la cellule donnée.

Dans tes notes : que faudrait-il encore livrer avec ce fichier pour qu'une autre équipe s'en serve (pense à la fiche modèle du ch. 23) ? Pourquoi `weights_only=True` ?

In [ ]:
# Two broken generators, for the check: your tests must catch them
def broken_float_35(path):
    saved = torch.load(path, weights_only=True)
    layers, n_in = [], saved["n_latent"]
    for n_out in reversed(saved["hidden"]):
        layers += [nn.Linear(n_in, n_out), nn.ReLU()]
        n_in = n_out
    decoder = nn.Sequential(*layers, nn.Linear(n_in, 784))
    decoder.load_state_dict(saved["state_dict"])
    decoder.eval()

    def generate(n, seed=0):
        z = torch.randn(n, saved["n_latent"], generator=torch.Generator().manual_seed(seed))
        with torch.inference_mode():
            return torch.sigmoid(decoder(z)).numpy().reshape(n, 28, 28)          # floats in [0, 1], not uint8

    return generate


def broken_seed_35(path):
    saved = torch.load(path, weights_only=True)
    layers, n_in = [], saved["n_latent"]
    for n_out in reversed(saved["hidden"]):
        layers += [nn.Linear(n_in, n_out), nn.ReLU()]
        n_in = n_out
    decoder = nn.Sequential(*layers, nn.Linear(n_in, 784))
    decoder.load_state_dict(saved["state_dict"])
    decoder.eval()

    def generate(n, seed=0):
        z = torch.randn(n, saved["n_latent"], generator=torch.Generator().manual_seed(0))   # the seed is ignored
        with torch.inference_mode():
            images = torch.round(torch.sigmoid(decoder(z)) * 255)
        return images.numpy().astype(np.uint8).reshape(n, 28, 28)

    return generate

In [ ]:
def export_decoder_35(vae, path):
    """Save what a generator needs, with torch.save, in the file `path`: the dict {"n_latent": vae.n_latent,
    "hidden": list(vae.hidden), "state_dict": vae.decoder.state_dict()} (the decoder only)."""
    raise NotImplementedError("export_decoder_35")


def load_generator_35(path):
    """Read the file of export_decoder_35 (torch.load(path, weights_only=True)), rebuild the decoder (the layers of
    25.30: Linear(n_latent, hidden[-1]) + ReLU, ..., Linear(hidden[0], 784)), load its weights, switch it to eval
    mode, and return a function generate(n, seed=0): z = torch.randn(n, n_latent,
    generator=torch.Generator().manual_seed(seed)), images = sigmoid(decoder(z)) * 255, rounded (torch.round),
    as a NumPy uint8 array (n, 28, 28). Only torch, nn and numpy: nothing from this notebook."""
    raise NotImplementedError("load_generator_35")


def test_generator_shape_35():
    """Your test: generate(5) gives 5 images of 28 x 28 in uint8 (use GEN_PATH, the path of the exported file)."""
    raise NotImplementedError("test_generator_shape_35")


def test_generator_seed_35():
    """Your test: the same seed gives the same images, two different seeds give different images."""
    raise NotImplementedError("test_generator_seed_35")

In [ ]:
GEN_PATH = str(WORK_DIR / "generator_35.pt")
with checked("25.35"):
    export_decoder_35(VAE_C.cpu(), GEN_PATH)
    VAE_C.to(DEVICE)
    generate_35 = load_generator_35(GEN_PATH)
    if returned("25.35", "load_generator_35", generate_35):
        images_35 = generate_35(6, seed=1)
        z_35 = torch.randn(6, 20, generator=torch.Generator().manual_seed(1))
        want_35 = np.round(vae_images(VAE_C, z_35.numpy()) * 255).astype(np.uint8).reshape(6, 28, 28)
        verdict("25.35", isinstance(images_35, np.ndarray) and images_35.dtype == np.uint8 and images_35.shape == (6, 28, 28)
                and np.abs(images_35.astype(int) - want_35.astype(int)).max() <= 1,
                "le générateur rechargé rend les images du décodeur de VAE_C, en uint8 (6, 28, 28).",
                f"attendu un tableau uint8 (6, 28, 28) égal aux images du décodeur de VAE_C pour la graine 1, reçu "
                f"{type(images_35).__name__} {shape_text(images_35)}.")
        show_rows([generate_35(10, seed=0) / 255.0], ["generate(10)"])
    PRE_35 = f"import torch\nimport torch.nn as nn\nGEN_PATH = {GEN_PATH!r}"
    TESTS_35 = [test_generator_shape_35, test_generator_seed_35]
    import inspect
    if all("NotImplementedError" not in inspect.getsource(test) for test in TESTS_35):
        print("Tes tests, sur ton générateur (ils doivent passer) :")
        result_35 = wb.run_pytest(TESTS_35, subject=load_generator_35, preamble=PRE_35)
        verdict("25.35", result_35.ok, "tes tests passent sur ton générateur.", "tes tests doivent passer sur ton générateur.")
        for broken_35, why_35 in [(broken_float_35, "un générateur qui rend des flottants dans [0, 1]"),
                                  (broken_seed_35, "un générateur qui ignore la graine")]:
            caught_35 = not wb.run_pytest(TESTS_35, subject=broken_35, name="load_generator_35", preamble=PRE_35, quiet=True).ok
            verdict("25.35", caught_35, f"tes tests attrapent {why_35}.", f"tes tests laissent passer {why_35}.")
    else:
        print("⏳ Ex 25.35 : écris tes deux tests.")

### Ex 25.36 — β-VAE : échanger la netteté contre la régularité 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer le compromis entre reconstruction et KL quand $\beta$ varie, et ce qu'il change aux images générées.  
**Prérequis :** Ex 25.30 · Ex 25.34 · fiche (Au-delà du livre : β-VAE) · **Fil rouge :** Fashion-MNIST · **Parcours :** M, C

Le $\beta$ de la loss règle le poids de la KL. Tu entraînes le même VAE (16 latents) sur Fashion-MNIST avec $\beta$ = 0,5, 1, 4 et 16, et tu compares la reconstruction, la KL, le nombre de latents actifs, les reconstructions et les images générées.

**Contrat** : `beta_sweep_36(betas, epochs, seed=2536, x=None, n_latent=16)` : pour chaque $\beta$ de `betas`, `torch.manual_seed(seed)`, `model = VAE(n_latent)`, `train_vae(model, x, epochs, beta=beta, seed=seed)` (avec `x = X_FASHION` si `x` vaut `None`), puis `rec, kl, per_latent = vae_eval(model, XF_TEST)` ; elle rend `{beta: {"rec": ..., "kl": ..., "active": le nombre de latents dont la KL moyenne dépasse 0,05 nat, "model": model}}`.

La vérification compare ta fonction à un entraînement fait à la main sur un cas d'essai. La cellule suivante lance les quatre entraînements (4 epochs en mode rapide, une trentaine de secondes), trace la reconstruction en fonction de la KL, et montre les reconstructions et des tirages pour chaque $\beta$.

Dans tes notes : comment la reconstruction et la KL évoluent-elles avec $\beta$ ? Que deviennent les images générées, et les reconstructions ? Combien de latents restent actifs à $\beta = 16$, et qu'est-ce que cela dit du code ? Quel $\beta$ choisirais-tu pour un générateur, et pour un détecteur d'anomalies ?

In [ ]:
def beta_sweep_36(betas, epochs, seed=2536, x=None, n_latent=16):
    """For each beta of `betas`: torch.manual_seed(seed), model = VAE(n_latent), train_vae(model, x, epochs,
    beta=beta, seed=seed) (x = X_FASHION if None), then rec, kl, per_latent = vae_eval(model, XF_TEST).
    Returns {beta: {"rec": rec, "kl": kl, "active": the number of latents whose mean KL is > 0.05 nat,
    "model": model}}."""
    raise NotImplementedError("beta_sweep_36")

In [ ]:
with checked("25.36"):
    tiny_36 = beta_sweep_36([2.0], 1, seed=4, x=X_FASHION[:512], n_latent=4)
    if returned("25.36", "beta_sweep_36", tiny_36):
        torch.manual_seed(4)
        ref_36 = VAE(4)
        train_vae(ref_36, X_FASHION[:512], 1, beta=2.0, seed=4)
        rec_36, kl_36, per_36 = vae_eval(ref_36, XF_TEST)
        row_36 = tiny_36.get(2.0, {}) if isinstance(tiny_36, dict) else {}
        verdict("25.36", {"rec", "kl", "active", "model"} <= set(row_36) and abs(row_36["rec"] - rec_36) < 1e-2
                and abs(row_36["kl"] - kl_36) < 1e-3 and row_36["active"] == int(np.sum(per_36 > 0.05)),
                f"beta_sweep_36 sur un cas d'essai : rec {rec_36:.1f}, kl {kl_36:.2f}, {int(np.sum(per_36 > 0.05))} latents actifs.",
                f"attendu {{2.0: {{'rec': {rec_36:.1f}, 'kl': {kl_36:.2f}, 'active': {int(np.sum(per_36 > 0.05))}, 'model': ...}}}}, "
                f"reçu {({k: v for k, v in row_36.items() if k != 'model'})!r:.200}.")

In [ ]:
with checked("25.36"):
    BETAS_36 = (0.5, 1.0, 4.0, 16.0)
    EPOCHS_36 = 4 if FAST_MODE else 20
    sweep_36 = beta_sweep_36(BETAS_36, EPOCHS_36)
    for beta_36, row_36 in sweep_36.items():
        print(f"beta = {beta_36:4.1f} · reconstruction {row_36['rec']:6.1f} nats · KL {row_36['kl']:5.1f} nats · "
              f"{row_36['active']:2d} latents actifs sur 16")
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.plot([r["kl"] for r in sweep_36.values()], [r["rec"] for r in sweep_36.values()], "o-")
    for beta_36, row_36 in sweep_36.items():
        ax.annotate(f"β = {beta_36:g}", (row_36["kl"], row_36["rec"]), fontsize=8)
    ax.set_xlabel("KL (nats)")
    ax.set_ylabel("reconstruction (nats)")
    plt.show()
    z_36 = np.random.default_rng(36).standard_normal((10, 16))
    show_rows([XF_TEST[:10]] + [vae_images(r["model"], encode_mu(r["model"], XF_TEST[:10])) for r in sweep_36.values()],
              ["originaux"] + [f"β = {b:g}" for b in BETAS_36], "reconstructions (décodées de mu)", size=0.9)
    show_rows([vae_images(r["model"], z_36) for r in sweep_36.values()], [f"β = {b:g}" for b in BETAS_36],
              "codes tirés de N(0, I)", size=0.9)

### Ex 25.37 — Détecter des anomalies par l'erreur de reconstruction 🔬 ★★★ ⏱️ 45 min
**Objectif :** utiliser l'erreur de reconstruction comme score d'anomalie, la mesurer par la ROC-AUC, et en voir la limite.  
**Prérequis :** Ex 25.24 · Ex 25.17 · ch. 3 (ROC-AUC, `mylearn.metrics`) · fiche (Au-delà du livre : anomalies) · **Fil rouge :** MNIST · **Parcours :** R, C

Un autoencodeur entraîné sur des données « normales » reconstruit mal ce qu'il n'a jamais vu : son erreur de reconstruction est un **score d'anomalie**. Tu entraînes `make_deep_ae(20)` sur tous les chiffres **sauf les 7**, puis tu mesures s'il repère les 7, les vêtements, et des chiffres très pâles.

**Contrat**
- `fit_without_37(digit, epochs, seed=2537)` crée `make_deep_ae(20)` juste après `torch.manual_seed(seed)` et l'entraîne par `train_ae(model, x, epochs, seed=seed)` sur les images de `X_MNIST` qui ne sont **pas** `digit` ; elle rend le modèle ;
- `scores_37(model, x)` rend le score de chaque image : la MSE entre l'image et sa reconstruction ;
- `auc_37(normal, anomalies)` rend la ROC-AUC des scores (un grand score = plus anormal) avec **ta** `mylearn.metrics.roc_auc` (ch. 3), le label 0 pour les scores de `normal` et 1 pour ceux de `anomalies`.

La vérification compare ta ROC-AUC à celle de scikit-learn et tes scores à un calcul direct. La cellule suivante entraîne le modèle sans les 7 (5 epochs en mode rapide), puis calcule trois ROC-AUC et trace les histogrammes des scores.

Dans tes notes : les 7 sont-ils faciles à repérer, et les vêtements ? Que se passe-t-il avec les chiffres pâles, et pourquoi est-ce une limite de toute détection par reconstruction (25.Q11) ? Comment choisirais-tu un seuil en pratique, et que mesurerais-tu (ch. 3) ?

In [ ]:
def fit_without_37(digit, epochs, seed=2537):
    """make_deep_ae(20), created just after torch.manual_seed(seed), trained by train_ae(model, x, epochs, seed=seed)
    on the images of X_MNIST that are NOT `digit` (Y_MNIST != digit). Returns the model."""
    raise NotImplementedError("fit_without_37")


def scores_37(model, x):
    """The anomaly score of each image: the MSE between the image and its reconstruction (an array of len(x))."""
    raise NotImplementedError("scores_37")


def auc_37(normal, anomalies):
    """The ROC-AUC of the scores (a larger score = more anomalous): mylearn.metrics.roc_auc with the label 0 for the
    scores of `normal` and 1 for those of `anomalies`."""
    raise NotImplementedError("auc_37")

In [ ]:
with checked("25.37"):
    from sklearn.metrics import roc_auc_score
    value_37 = auc_37(np.array([0.1, 0.2, 0.4]), np.array([0.3, 0.5]))
    if returned("25.37", "auc_37", value_37):
        same("25.37", "auc_37([0.1, 0.2, 0.4], [0.3, 0.5]) contre sklearn.metrics.roc_auc_score", [value_37],
             [roc_auc_score([0, 0, 0, 1, 1], [0.1, 0.2, 0.4, 0.3, 0.5])])
    model_37 = fit_without_37(7, 1, seed=5)
    if returned("25.37", "fit_without_37", model_37):
        torch.manual_seed(5)
        ref_37 = make_deep_ae(20)
        train_ae(ref_37, X_MNIST[Y_MNIST != 7], 1, seed=5)
        same("25.37", "fit_without_37(7, 1, seed=5) : ses reconstructions de 3 chiffres, contre un entraînement fait à la "
             "main sans les 7", reconstruct(model_37, XM_TEST[:3]), reconstruct(ref_37, XM_TEST[:3]), rtol=1e-4, atol=1e-5)
    s_37 = scores_37(AE_DEEP, XM_TEST[:4])
    if returned("25.37", "scores_37", s_37):
        same("25.37", "scores_37(AE_DEEP, 4 chiffres)", s_37, mse_per_image(reconstruct(AE_DEEP, XM_TEST[:4]), XM_TEST[:4]),
             rtol=1e-5)

In [ ]:
with checked("25.37"):
    EPOCHS_37 = 5 if FAST_MODE else 15
    model_37 = fit_without_37(7, EPOCHS_37)
    normal_37 = scores_37(model_37, XM_TEST[YM_TEST != 7])
    sevens_37 = scores_37(model_37, XM_TEST[YM_TEST == 7])
    clothes_37 = scores_37(model_37, XF_TEST)
    faint_37 = scores_37(model_37, 0.15 * XM_TEST[YM_TEST != 7][:200])          # digits drawn very faintly
    print(f"ROC-AUC · les 7 contre les autres chiffres {auc_37(normal_37, sevens_37):.3f} · les vêtements contre les "
          f"chiffres {auc_37(normal_37, clothes_37):.3f} · des chiffres très pâles contre les chiffres {auc_37(normal_37, faint_37):.3f}")
    fig, ax = plt.subplots(figsize=(8, 3))
    bins_37 = np.linspace(0, np.quantile(np.concatenate([normal_37, sevens_37, clothes_37]), 0.99), 60)
    for values_37, name_37 in [(normal_37, "chiffres (pas de 7)"), (sevens_37, "7"), (clothes_37, "vêtements"),
                               (faint_37, "chiffres pâles")]:
        ax.hist(values_37, bins=bins_37, alpha=0.5, density=True, label=name_37)
    ax.set_xlabel("score d'anomalie (MSE de reconstruction)")
    ax.legend()
    plt.show()

### Ex 25.38 — Un VAE entièrement en NumPy avec mylearn 🔨 ★★★★ ⏱️ 150 min
**Objectif :** assembler un VAE complet avec ta librairie (couches, passe arrière, Adam, `vae.py`) et vérifier sa passe arrière.  
**Prérequis :** Ex 25.28 · Ex 25.29 · ch. 18 (Ex 18.14, `mlp_backward`) · ch. 19 (Adam) · fiche §25.9.2 · **Fil rouge :** MNIST · **mylearn :** `vae.py, nn/layers.py, nn/backward.py, optim.py` · **Parcours :** M, C

Tout ce qu'il faut pour un VAE est déjà dans ta librairie : `init_mlp` et `mlp_forward` (ch. 16), `binary_cross_entropy_with_logits` et `mlp_backward` (ch. 18), `Adam` (ch. 19), et `vae.py`. Tu l'assembles **sans nouvelle fonction de librairie** : un encodeur `784 → hidden → 2d` (ReLU au milieu ; les $d$ premières sorties sont $\boldsymbol{\mu}$, les $d$ suivantes $\log\boldsymbol{\sigma}^2$), un décodeur `d → hidden → 784` (des logits).

**Contrat**
- `init_numpy_vae_38(n_latent=8, hidden=128, seed=0)` rend `{"enc": init_mlp([784, hidden, 2 * n_latent], rng=rng), "dec": init_mlp([n_latent, hidden, 784], rng=rng)}`, avec **un seul** `rng = np.random.default_rng(seed)`, l'encodeur d'abord ;
- `numpy_vae_loss_38(params, x, eps)` fait une passe avant et une passe arrière sur le batch `x` `(n, 784)`, avec le bruit `eps` `(n, d)` **donné** : `z = mu + exp(0.5 log_var) * eps` ; `rec` = la cross-entropy binaire des logits **sommée** sur les pixels et moyennée sur le batch ; `kl` = la moyenne sur le batch de `gaussian_kl` ; `loss = rec + kl` ; elle rend `(loss, rec, kl, grads)`, avec `grads = {"enc": [(dW, db), ...], "dec": [(dW, db), ...]}` comme `params` ; la ReLU et sa dérivée sont `_relu_38` et `_relu_derivative_38` (données dans la cellule) ;
- `train_numpy_vae_38(params, x, epochs, lr=1e-3, batch_size=128, seed=0)` entraîne `params` sur place avec `mylearn.optim.Adam` sur la liste à plat `[W1, b1, W2, b2, ...]` de l'encodeur puis du décodeur ; `rng = np.random.default_rng(seed)` : à chaque epoch `rng.permutation(N)`, puis pour chaque mini-batch `eps = rng.standard_normal((n, d))` ; elle rend la liste des losses moyennes de chaque epoch.

La vérification construit le même petit VAE en PyTorch (`float64`, poids copiés, même `eps`) et compare ta loss et **tous** tes gradients à `torch.autograd`, avant tout entraînement. La cellule suivante entraîne ton VAE NumPy (5 epochs sur 5 000 chiffres en mode rapide) et décode des codes tirés de $\mathcal{N}(\mathbf{0}, \mathbf{I})$.

Dans tes notes : par où le gradient arrive-t-il sur $\boldsymbol{\mu}$ et sur $\log\boldsymbol{\sigma}^2$ (deux chemins) ? Pourquoi la vérification contre `torch.autograd` passe-t-elle **avant** l'entraînement ? Combien de temps prend une epoch, face au VAE PyTorch ?

In [ ]:
def _relu_38(z):
    return np.maximum(z, 0.0)


def _relu_derivative_38(z):
    return (z > 0).astype(float)


def init_numpy_vae_38(n_latent=8, hidden=128, seed=0):
    """{"enc": init_mlp([784, hidden, 2 * n_latent], rng=rng), "dec": init_mlp([n_latent, hidden, 784], rng=rng)},
    with ONE rng = np.random.default_rng(seed), the encoder first (mylearn.nn.layers.init_mlp)."""
    raise NotImplementedError("init_numpy_vae_38")


def numpy_vae_loss_38(params, x, eps):
    """One forward and backward pass of the NumPy VAE on the batch x (n, 784), with the noise eps (n, d) given.
    Returns (loss, rec, kl, grads), grads = {"enc": [(dW, db), ...], "dec": [(dW, db), ...]} like params."""
    raise NotImplementedError("numpy_vae_loss_38")


def train_numpy_vae_38(params, x, epochs, lr=1e-3, batch_size=128, seed=0):
    """Train params in place with mylearn.optim.Adam on the flat list [W1, b1, W2, b2, ...] of the encoder then the
    decoder; rng = np.random.default_rng(seed): each epoch, rng.permutation(N), then for each mini-batch
    eps = rng.standard_normal((n, d)). Returns the list of the mean losses of each epoch."""
    raise NotImplementedError("train_numpy_vae_38")

In [ ]:
with checked("25.38"):
    params_38 = init_numpy_vae_38(n_latent=3, hidden=16, seed=1)
    if returned("25.38", "init_numpy_vae_38", params_38):
        shapes_38 = {k: [(W.shape, b.shape) for W, b in v] for k, v in params_38.items()}
        verdict("25.38", shapes_38 == {"enc": [((784, 16), (16,)), ((16, 6), (6,))], "dec": [((3, 16), (16,)), ((16, 784), (784,))]},
                "init_numpy_vae_38(3, 16) : les formes attendues.", f"formes reçues : {shapes_38}.")
        x_38 = X_MNIST[:5].astype(np.float64)
        eps_38 = np.random.default_rng(2).standard_normal((5, 3))
        result_38 = numpy_vae_loss_38(params_38, x_38, eps_38)
        if returned("25.38", "numpy_vae_loss_38", result_38):
            loss_38, rec_38, kl_38, grads_38 = result_38
            # the same VAE in PyTorch, float64, with copies of the weights: its autograd gives the reference
            tp_38 = {k: [(torch.tensor(W, requires_grad=True), torch.tensor(b, requires_grad=True)) for W, b in v]
                     for k, v in params_38.items()}
            def _mlp_38(a, layers):
                for i, (W, b) in enumerate(layers):
                    a = a @ W + b
                    if i < len(layers) - 1:
                        a = torch.relu(a)
                return a
            xt_38 = torch.tensor(x_38)
            h_38 = _mlp_38(xt_38, tp_38["enc"])
            mu_t38, lv_t38 = h_38[:, :3], h_38[:, 3:]
            logits_t38 = _mlp_38(mu_t38 + torch.exp(0.5 * lv_t38) * torch.tensor(eps_38), tp_38["dec"])
            rec_t38 = F.binary_cross_entropy_with_logits(logits_t38, xt_38, reduction="sum") / 5
            kl_t38 = (0.5 * (mu_t38 ** 2 + torch.exp(lv_t38) - 1 - lv_t38)).sum(dim=1).mean()
            (rec_t38 + kl_t38).backward()
            same("25.38", "(loss, rec, kl) contre le même VAE en PyTorch", [loss_38, rec_38, kl_38],
                 [float(rec_t38 + kl_t38), float(rec_t38), float(kl_t38)], rtol=1e-8)
            for part_38 in ("enc", "dec"):
                for i_38, ((dW_38, db_38), (W_t, b_t)) in enumerate(zip(grads_38[part_38], tp_38[part_38])):
                    same("25.38", f"dW de la couche {i_38} de {part_38} contre torch.autograd", dW_38, W_t.grad.numpy(),
                         rtol=1e-6, atol=1e-10)
                    same("25.38", f"db de la couche {i_38} de {part_38} contre torch.autograd", db_38, b_t.grad.numpy(),
                         rtol=1e-6, atol=1e-10)

In [ ]:
with checked("25.38"):
    EPOCHS_38 = 5 if FAST_MODE else 30
    params_38 = init_numpy_vae_38(n_latent=8, hidden=128, seed=2538)
    start_38 = time.perf_counter()
    losses_38 = train_numpy_vae_38(params_38, X_MNIST[:5000] if FAST_MODE else X_MNIST, EPOCHS_38, seed=2538)
    print(f"{EPOCHS_38} epochs en {time.perf_counter() - start_38:.0f} s · loss par epoch : " + " ".join(f"{v:.1f}" for v in losses_38))
    verdict("25.38", losses_38[-1] < losses_38[0] - 20, "la loss baisse : ton VAE en NumPy apprend.",
            "la loss devrait baisser d'au moins 20 nats : vérifie les gradients (cellule précédente).")
    z_38 = np.random.default_rng(38).standard_normal((12, 8))
    logits_38 = mylearn.nn.layers.mlp_forward(z_38, params_38["dec"], activation=lambda a: np.maximum(a, 0.0))
    show_rows([1 / (1 + np.exp(-logits_38))], ["N(0, I) décodés"], "ton VAE NumPy comme générateur")

### Ex 25.39 — Défi débruitage Fashion-MNIST : le meilleur PSNR sous contrainte 🏆 ★★★★ ⏱️ 120 min
**Objectif :** concevoir et entraîner un débruiteur de vêtements qui atteint un PSNR fixé, sous contraintes de taille et de temps.  
**Prérequis :** Ex 25.26 · Ex 25.27 · fiche §25.8 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Le défi : débruiter des vêtements abîmés par un bruit gaussien d'écart-type 0,3 (écrêté dans $[0, 1]$), avec un PSNR moyen d'au moins `GOAL_39` dB, d'abord sur les 1 000 vêtements de validation de la partie B, puis sur 1 000 **autres** vêtements de test, cachés.

**Règles**
- `make_denoiser_39()` rend un `AutoEncoder` (des lignes de 784 pixels en entrée, des lignes de 784 pixels dans $[0, 1]$ en sortie) d'au plus 100 000 paramètres ;
- `train_39(model, x_clean)` l'entraîne sur les vêtements propres `X_FASHION` (tous, ou une partie) ; tout est permis : architecture, boucle (`train_denoiser` ou la tienne, avec le bruit de l'énoncé), optimiseur, nombre d'epochs, augmentation ; l'entraînement doit tenir en 300 s en mode rapide (900 s en mode complet) ;
- règles d'honneur : n'utilise ni `XF_TEST` ni `VAL_NOISY_B` pour entraîner, et ne règle rien sur le test caché ; il n'est révélé qu'**une fois par session**, et le résultat est figé.

La vérification compte les paramètres, chronomètre `train_39`, mesure le PSNR de validation, puis, si l'objectif est atteint, révèle le test. Le débruiteur de 25.26 (32 filtres, 3 epochs sur 10 000 images) n'y arrive pas : à toi de trouver ce qui compte.

Dans tes notes : qu'as-tu essayé, dans quel ordre, et qu'est-ce qui a le plus aidé (largeur, nombre d'images, epochs, learning rate, architecture sans pooling) ? Combien de secondes ton entraînement laisse-t-il de marge ?

In [ ]:
# The challenge: the validation (public) and the test (hidden) clothes, with a fixed noise of sigma = 0.3
TEST_CLEAN_39 = XF_TEST[1000:2000]
TEST_NOISY_39 = np.clip(TEST_CLEAN_39 + SIGMA_B * np.random.default_rng(2539).standard_normal(TEST_CLEAN_39.shape),
                        0, 1).astype(np.float32)
GOAL_39 = 19.3                    # dB on the validation, then on the test
MAX_PARAMS_39 = 100_000
MAX_SECONDS_39 = 300 if FAST_MODE else 900
print(f"objectif : PSNR ≥ {GOAL_39} dB · au plus {MAX_PARAMS_39:,d} paramètres · au plus {MAX_SECONDS_39} s d'entraînement")

In [ ]:
def make_denoiser_39():
    """Your denoiser: an AutoEncoder (rows of 784 pixels in, rows of 784 pixels in [0, 1] out), at most
    MAX_PARAMS_39 parameters."""
    raise NotImplementedError("make_denoiser_39")


def train_39(model, x_clean):
    """Train `model` in place to remove a gaussian noise of standard deviation SIGMA_B (clipped to [0, 1]) from the
    clean images x_clean (X_FASHION): any loop, optimizer or number of epochs, within MAX_SECONDS_39 seconds."""
    raise NotImplementedError("train_39")

In [ ]:
with checked("25.39"):
    torch.manual_seed(2539)
    model_39 = make_denoiser_39()
    if returned("25.39", "make_denoiser_39", model_39):
        count_39 = n_params(model_39)
        if count_39 > MAX_PARAMS_39:
            print(f"❌ Ex 25.39 : {count_39:,d} paramètres, plus que les {MAX_PARAMS_39:,d} permis.")
            raise StopCheck
        start_39 = time.perf_counter()
        train_39(model_39.to(DEVICE), X_FASHION)
        seconds_39 = time.perf_counter() - start_39
        val_39 = psnr(VAL_CLEAN_B, reconstruct(model_39, VAL_NOISY_B))
        print(f"{count_39:,d} paramètres · entraînement {seconds_39:.0f} s · PSNR de validation {val_39:.2f} dB "
              f"(images bruitées : {psnr(VAL_CLEAN_B, VAL_NOISY_B):.2f} dB)")
        if seconds_39 > MAX_SECONDS_39:
            print(f"❌ Ex 25.39 : l'entraînement a duré plus de {MAX_SECONDS_39} s.")
        elif val_39 < GOAL_39:
            print(f"❌ Ex 25.39 : il faut au moins {GOAL_39} dB sur la validation avant de révéler le test.")
        else:
            REVEALED_39 = globals().setdefault("REVEALED_39", psnr(TEST_CLEAN_39, reconstruct(model_39, TEST_NOISY_39)))
            print(f"test (révélé une fois par session ; un nouvel essai affiche la même valeur) : {REVEALED_39:.2f} dB")
            verdict("25.39", REVEALED_39 >= GOAL_39, f"défi réussi : {REVEALED_39:.2f} dB sur le test caché.",
                    f"sur le test caché, {REVEALED_39:.2f} dB : moins que les {GOAL_39} dB visés.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu expliquer le rôle du goulot, et pourquoi un autoencodeur entraîné sur une seule image ne comprime rien ?
2. Sais-tu dire ce que l'autoencodeur linéaire a en commun avec l'ACP, et ce qu'il ne retrouve pas ?
3. Sais-tu écrire la reparamétrisation et la KL gaussienne, et expliquer pourquoi un VAE sait générer ?

**Pour aller plus loin** : l'introduction aux VAE de Kingma et Welling (2019), citée dans la fiche. La suite : le ch. 26, l'apprentissage par renforcement, puis les GAN au ch. 27.